In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [2]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [3]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper Functions

In [4]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

def demand_driver_realign_pskus(data, channel):
    """
    Realign the old pskus to new pskus and return updated data.

    Args:
        data: pandas dataframe
        - master dataframe having all the pskus
    
    Return:
        data: pandas dataframe
        - dataframe 
    """
    realignment_data = realignment_df.copy()
    realignment_data.columns = realignment_data.columns.str.lower()
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "ALL")
    ]

    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for grp, grp_data in realignment_data.groupby(by=["psku old", "asm"]):
        old_psku, old_asm = grp
        new_psku = grp_data["psku new"].values[0]
        if old_asm != "ALL":
            condition = (data["parent_material_code"] == old_psku) & (
                data["asm_area_code"] == old_asm
            )
        else:
            condition = data["parent_material_code"] == old_psku

        data.loc[condition, "parent_material_code"] = new_psku

    return data

### Aggregation

In [363]:
data_query = f"""
    select * from TRN_MIL_DF_HEURISTICS_OUTPUT 
"""
df_heuristics = pd.read_sql(data_query, dev_conn)
df_heuristics

,CHANNEL,PORTFOLIO,BRAND,BRAND CLASS,RUN_MONTH,M MONTH,MONTH,ASM,DEPOT,PSKU,PROPHET VOL,RF_VOL,PROPHET HEURISTIC VOL,RF HEURISTIC VOL
0,ECOM,Hair Oils,ADV-AHO-R,B,2025-10-31,M,2025-10-31,BCE1,D231,718589,0.000000,3.085714,4.500000,4.50
1,ECOM,Hair Oils,ADV-AHO-R,B,2025-10-31,M+1,2025-11-30,BCE1,D231,718589,5.322387,3.150000,5.322387,4.50
2,ECOM,Hair Oils,ADV-AHO-R,B,2025-10-31,M+2,2025-12-31,BCE1,D231,718589,4.054349,3.600000,4.500000,4.50
3,ECOM,Hair Oils,ADV-AHO-R,B,2025-10-31,M+3,2026-01-31,BCE1,D231,718589,6.112519,3.600000,6.112519,4.50
4,ECOM,Hair Oils,ADV-AHO-R,B,2025-10-31,M+4,2026-02-28,BCE1,D231,718589,4.983325,4.178571,4.983325,4.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282199,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+1,2025-12-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
282200,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+2,2026-01-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
282201,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+3,2026-02-28,QCW2,D463,810125,NaN,NaN,0.640000,0.64
282202,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+4,2026-03-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64


In [364]:
df_heuristics.columns = df_heuristics.columns.str.lower()
df_heuristics['run_month'] = pd.to_datetime(df_heuristics['run_month'])
df_heuristics['month'] = pd.to_datetime(df_heuristics['month'])
df_heuristics


,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol
0,ECOM,Hair Oils,ADV-AHO-R,B,2025-10-31,M,2025-10-31,BCE1,D231,718589,0.000000,3.085714,4.500000,4.50
1,ECOM,Hair Oils,ADV-AHO-R,B,2025-10-31,M+1,2025-11-30,BCE1,D231,718589,5.322387,3.150000,5.322387,4.50
2,ECOM,Hair Oils,ADV-AHO-R,B,2025-10-31,M+2,2025-12-31,BCE1,D231,718589,4.054349,3.600000,4.500000,4.50
3,ECOM,Hair Oils,ADV-AHO-R,B,2025-10-31,M+3,2026-01-31,BCE1,D231,718589,6.112519,3.600000,6.112519,4.50
4,ECOM,Hair Oils,ADV-AHO-R,B,2025-10-31,M+4,2026-02-28,BCE1,D231,718589,4.983325,4.178571,4.983325,4.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282199,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+1,2025-12-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
282200,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+2,2026-01-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
282201,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+3,2026-02-28,QCW2,D463,810125,NaN,NaN,0.640000,0.64
282202,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+4,2026-03-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64


In [365]:
# delete_query = """
# DELETE FROM TRN_MIL_DF_SHARED
# WHERE "run_month" IN ('2025-12-31', '2026-01-31')
# """

# cur = dev_conn.cursor()
# cur.execute(delete_query)
# cur.close()


In [366]:
data_query = """
SELECT *
FROM TRN_MIL_DF_SHARED
   
"""

df = pd.read_sql(data_query, dev_conn)

In [367]:
df.columns = df.columns.str.lower()
df['run_month'] = pd.to_datetime(df['run_month'])
df['month'] = pd.to_datetime(df['month'])
df

,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,pred vol (roum)
0,GT,Saffola Oils,SAFF GOLD,A,2026-02-28,M+1,2026-03-31,AURG,D3A4,718288,3.702043
1,GT,Saffola Oils,SAFF GOLD,A,2026-02-28,M+2,2026-04-30,AURG,D3A4,718288,4.192877
2,GT,Saffola Oils,SAFF GOLD,A,2026-02-28,M+3,2026-05-31,AURG,D3A4,718288,3.864759
3,GT,Saffola Oils,SAFF GOLD,A,2026-02-28,M+4,2026-06-30,AURG,D3A4,718288,4.177644
4,GT,CNO,PCNO(R),A,2026-02-28,M+1,2026-03-31,AURG,D3A4,718297,25.549946
...,...,...,...,...,...,...,...,...,...,...,...
512645,MT,Foods,TRU_ELMNT,C,2026-01-31,M+1,2026-02-28,MCW2,D461,811100,0.016000
512646,MT,Foods,TRU_ELMNT,C,2026-01-31,M+2,2026-03-31,MCW2,D461,811100,0.016000
512647,MT,Foods,TRU_ELMNT,C,2026-01-31,M+3,2026-04-30,MCW2,D461,811100,0.016000
512648,MT,Foods,TRU_ELMNT,C,2026-01-31,M+4,2026-05-31,MCW2,D461,811100,0.016000


In [368]:
df[(df['month'] == '2026-04-30') & (df['channel']=='GT') & (df['run_month']=='2026-03-31')]['pred vol (roum)'].sum()


5346018.0036901245

In [369]:
# # make sure run_month is datetime
# df["run_month"] = pd.to_datetime(df["run_month"])
# upload_df["run_month"] = pd.to_datetime(upload_df["run_month"])

# # filter condition
# mask = (df["channel"] == "GT") & (df["run_month"] == "2026-01-31")

# # remove those rows
# df_filtered = df.loc[~mask]

# # concat upload_df
# final_df = pd.concat([df_filtered, upload_df], ignore_index=True)


In [370]:
df_heuristics.columns

Index(['channel', 'portfolio', 'brand', 'brand class', 'run_month', 'm month',
       'month', 'asm', 'depot', 'psku', 'prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol'],
      dtype='object')

In [371]:
df = df.merge(df_heuristics[['channel','asm', 'depot', 'psku', 'run_month', 'month','prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol']], on = ['channel','asm', 'depot', 'psku', 'run_month', 'month'],
       how = 'left')

In [372]:
df.isnull().sum()

channel                       0
portfolio                     0
brand                         0
brand class                 422
run_month                     0
m month                       0
month                         0
asm                           0
depot                         0
psku                          0
pred vol (roum)               0
prophet vol              306871
rf_vol                   306871
prophet heuristic vol    270591
rf heuristic vol         270591
dtype: int64

In [373]:
# df = df[df['Month'] == '2025-12-01']
# df

In [374]:
import numpy as np

df['final_channel'] = np.where(
    df['channel'].isin(['QCOM', 'GT']),
    df['channel'],
    np.where(
        df['channel'].isin(['ECOM', 'MT']) &
        df['asm'].astype(str).str.startswith('B'),
        'B2B',
        df['channel']
    )
)
df

,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,final_channel
0,GT,Saffola Oils,SAFF GOLD,A,2026-02-28,M+1,2026-03-31,AURG,D3A4,718288,3.702043,NaN,NaN,NaN,NaN,GT
1,GT,Saffola Oils,SAFF GOLD,A,2026-02-28,M+2,2026-04-30,AURG,D3A4,718288,4.192877,NaN,NaN,NaN,NaN,GT
2,GT,Saffola Oils,SAFF GOLD,A,2026-02-28,M+3,2026-05-31,AURG,D3A4,718288,3.864759,NaN,NaN,NaN,NaN,GT
3,GT,Saffola Oils,SAFF GOLD,A,2026-02-28,M+4,2026-06-30,AURG,D3A4,718288,4.177644,NaN,NaN,NaN,NaN,GT
4,GT,CNO,PCNO(R),A,2026-02-28,M+1,2026-03-31,AURG,D3A4,718297,25.549946,NaN,NaN,NaN,NaN,GT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
512645,MT,Foods,TRU_ELMNT,C,2026-01-31,M+1,2026-02-28,MCW2,D461,811100,0.016000,NaN,NaN,NaN,NaN,MT
512646,MT,Foods,TRU_ELMNT,C,2026-01-31,M+2,2026-03-31,MCW2,D461,811100,0.016000,NaN,NaN,NaN,NaN,MT
512647,MT,Foods,TRU_ELMNT,C,2026-01-31,M+3,2026-04-30,MCW2,D461,811100,0.016000,NaN,NaN,NaN,NaN,MT
512648,MT,Foods,TRU_ELMNT,C,2026-01-31,M+4,2026-05-31,MCW2,D461,811100,0.016000,NaN,NaN,NaN,NaN,MT


In [375]:
df.columns

Index(['channel', 'portfolio', 'brand', 'brand class', 'run_month', 'm month',
       'month', 'asm', 'depot', 'psku', 'pred vol (roum)', 'prophet vol',
       'rf_vol', 'prophet heuristic vol', 'rf heuristic vol', 'final_channel'],
      dtype='object')

In [376]:
df = df.groupby(['final_channel', 'portfolio', 'brand', 'brand class', 'run_month', 'm month',
       'month', 'asm', 'depot', 'psku'])[['pred vol (roum)', 'prophet vol',
       'rf_vol', 'prophet heuristic vol', 'rf heuristic vol']].sum().reset_index()#['Channel'].unique()
df.rename(columns = {'final_channel':'channel'}, inplace = True)
df

,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol
0,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D231,718474,0.027000,0.000000,0.029317,0.027000,0.029317
1,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D231,718475,0.162094,0.162094,0.061539,0.068033,0.068033
2,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D231,718476,0.156568,0.156568,0.066100,0.024700,0.031400
3,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D232,718475,0.000000,0.000000,0.001933,0.000000,0.000000
4,B2B,CNO,NHR-UTTAM,B,2025-10-31,M+1,2025-11-30,BCE1,D231,718474,0.009717,0.008834,0.008817,0.009717,0.027000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501296,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809807,0.000000,14.685752,2.232000,0.000000,0.000000
501297,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809850,0.000000,0.000000,0.120000,0.000000,0.000000
501298,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809865,0.000000,0.000000,0.746667,0.000000,0.000000
501299,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809866,0.000000,0.000000,0.000000,0.000000,0.000000


### Actuals

In [377]:
actuals_query = """
SELECT
    CM.channel_name,
    CM.asm_area_code,
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        channel_name, 
        asm_area_code,
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM ON MESR.distributor_code = CM.customer_code
WHERE
    month_date BETWEEN '2023-01-31' AND '2026-04-30' AND
    CM.channel_name IN ('MT', 'E-Commerce', 'Q-Commerce', 'GT')
GROUP BY 1, 2, 3, 4, 5, 6
ORDER BY 1, 2, 3, 4, 6
"""

actuals_df = pd.read_sql(
    actuals_query,
    prod_conn
)

In [378]:
actuals_df.columns = actuals_df.columns.str.lower()
actuals_df['month_date'] = pd.to_datetime(actuals_df['month_date'])

In [379]:
actuals_df.head()

,channel_name,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,E-Commerce,BCE1,D231,718297,PCNO(R),2023-01-31,0.000,0.000,0.808,0.076
1,E-Commerce,BCE1,D231,718297,PCNO(R),2023-02-28,0.229,0.525,0.774,0.229
2,E-Commerce,BCE1,D231,718297,PCNO(R),2023-03-31,0.364,0.549,0.724,0.364
3,E-Commerce,BCE1,D231,718297,PCNO(R),2023-04-30,0.057,0.189,0.282,0.057
4,E-Commerce,BCE1,D231,718297,PCNO(R),2023-05-31,-0.021,0.335,0.528,-0.021


In [380]:
actuals_df.duplicated(subset=['channel_name', 'asm_area_code', 'depot_code', 'parent_material_code','month_date']).sum()

383

In [381]:
# actuals_df['key'] =  actuals_df['asm_area_code'].astype(str) + '_' + actuals_df['depot_code'].astype(str) + '_' + actuals_df['parent_material_code'].astype(str) 
# actuals_df

In [382]:
# actuals_df[(actuals_df['channel_name'] == 'GT') & (actuals_df['month_date']=='2026-02-28')]['key'].nunique()

In [383]:
actuals_df['channel_name'] = actuals_df['channel_name'].replace({
    'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})

In [384]:
actuals_df

,channel_name,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,ECOM,BCE1,D231,718297,PCNO(R),2023-01-31,0.000,0.000000,0.8080,0.076
1,ECOM,BCE1,D231,718297,PCNO(R),2023-02-28,0.229,0.525000,0.7740,0.229
2,ECOM,BCE1,D231,718297,PCNO(R),2023-03-31,0.364,0.549000,0.7240,0.364
3,ECOM,BCE1,D231,718297,PCNO(R),2023-04-30,0.057,0.189000,0.2820,0.057
4,ECOM,BCE1,D231,718297,PCNO(R),2023-05-31,-0.021,0.335000,0.5280,-0.021
...,...,...,...,...,...,...,...,...,...,...
2276293,QCOM,QCW2,D463,811021,PADV_WIPS,2026-04-30,0.000,17.993646,0.0000,0.000
2276294,QCOM,QCW2,D463,811169,SW_SGPRF,2026-03-31,0.000,13.043478,0.0000,0.000
2276295,QCOM,QCW2,D463,811169,SW_SGPRF,2026-04-30,0.000,3.969082,0.0000,0.000
2276296,QCOM,QCW2,D463,811181,SAF_CDPRS,2026-04-30,0.096,0.000000,0.0000,0.096


In [385]:
import numpy as np

actuals_df['final_channel'] = np.where(
    actuals_df['channel_name'].isin(['QCOM', 'GT']),
    actuals_df['channel_name'],
    np.where(
        actuals_df['channel_name'].isin(['ECOM', 'MT']) &
        actuals_df['asm_area_code'].astype(str).str.startswith('B'),
        'B2B',
        actuals_df['channel_name']
    )
)
actuals_df

,channel_name,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,final_channel
0,ECOM,BCE1,D231,718297,PCNO(R),2023-01-31,0.000,0.000000,0.8080,0.076,B2B
1,ECOM,BCE1,D231,718297,PCNO(R),2023-02-28,0.229,0.525000,0.7740,0.229,B2B
2,ECOM,BCE1,D231,718297,PCNO(R),2023-03-31,0.364,0.549000,0.7240,0.364,B2B
3,ECOM,BCE1,D231,718297,PCNO(R),2023-04-30,0.057,0.189000,0.2820,0.057,B2B
4,ECOM,BCE1,D231,718297,PCNO(R),2023-05-31,-0.021,0.335000,0.5280,-0.021,B2B
...,...,...,...,...,...,...,...,...,...,...,...
2276293,QCOM,QCW2,D463,811021,PADV_WIPS,2026-04-30,0.000,17.993646,0.0000,0.000,QCOM
2276294,QCOM,QCW2,D463,811169,SW_SGPRF,2026-03-31,0.000,13.043478,0.0000,0.000,QCOM
2276295,QCOM,QCW2,D463,811169,SW_SGPRF,2026-04-30,0.000,3.969082,0.0000,0.000,QCOM
2276296,QCOM,QCW2,D463,811181,SAF_CDPRS,2026-04-30,0.096,0.000000,0.0000,0.096,QCOM


In [386]:
actuals_df = actuals_df.groupby(['final_channel', 'asm_area_code', 'depot_code', 'parent_material_code',
       'material_group_code', 'month_date'])[['pri_actuals_vol_rum',
       'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum().reset_index()
actuals_df

,final_channel,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,B2B,BCE1,D231,702478,PADV-HRCR,2024-09-30,0.000,0.000000,0.0000,0.000
1,B2B,BCE1,D231,702478,PADV-HRCR,2024-10-31,0.000,0.000000,0.0000,0.000
2,B2B,BCE1,D231,702478,PADV-HRCR,2024-12-31,0.000,0.000000,0.0000,0.000
3,B2B,BCE1,D231,705148,NHR-UTTAM,2024-09-30,0.000,0.000000,0.0000,0.000
4,B2B,BCE1,D231,705148,NHR-UTTAM,2024-10-31,0.000,0.000000,0.0000,0.000
...,...,...,...,...,...,...,...,...,...,...
2203884,QCOM,QCW2,D463,811021,PADV_WIPS,2026-04-30,0.000,17.993646,0.0000,0.000
2203885,QCOM,QCW2,D463,811169,SW_SGPRF,2026-03-31,0.000,13.043478,0.0000,0.000
2203886,QCOM,QCW2,D463,811169,SW_SGPRF,2026-04-30,0.000,3.969082,0.0000,0.000
2203887,QCOM,QCW2,D463,811181,SAF_CDPRS,2026-04-30,0.096,0.000000,0.0000,0.096


In [387]:
vol_cols = [
    'pri_actuals_vol_rum',
    'pri_apo_plan_vol_rum',
    'sec_apo_plan_vol_rum',
    'sec_actuals_vol_rum'
]

actuals_df[vol_cols] = actuals_df[vol_cols].clip(lower=0)

In [388]:
actuals_df.rename(columns = {'final_channel':'channel_name'}, inplace = True)

In [389]:
tmp_df = pd.DataFrame()

for c in ['GT', 'MT', 'ECOM', 'QCOM','B2B']:
    tmp2_df = actuals_df[actuals_df['channel_name'] == c]
    tmp2_df = demand_driver_realign_pskus(tmp2_df, channel=c)
    tmp_df = pd.concat([tmp_df, tmp2_df], ignore_index=True)

In [390]:
tmp_df.duplicated(subset=['channel_name', 'asm_area_code', 'depot_code', 'parent_material_code','month_date']).sum()

14628

In [391]:
tmp_df['month_date'].unique()

<DatetimeArray>
['2023-02-28 00:00:00', '2023-03-31 00:00:00', '2023-04-30 00:00:00',
 '2023-05-31 00:00:00', '2023-06-30 00:00:00', '2023-07-31 00:00:00',
 '2023-08-31 00:00:00', '2023-09-30 00:00:00', '2023-10-31 00:00:00',
 '2023-11-30 00:00:00', '2023-12-31 00:00:00', '2024-01-31 00:00:00',
 '2024-02-29 00:00:00', '2024-03-31 00:00:00', '2024-04-30 00:00:00',
 '2024-05-31 00:00:00', '2024-06-30 00:00:00', '2024-07-31 00:00:00',
 '2024-08-31 00:00:00', '2024-09-30 00:00:00', '2024-10-31 00:00:00',
 '2024-11-30 00:00:00', '2024-12-31 00:00:00', '2025-01-31 00:00:00',
 '2025-02-28 00:00:00', '2025-03-31 00:00:00', '2025-04-30 00:00:00',
 '2025-05-31 00:00:00', '2025-06-30 00:00:00', '2025-07-31 00:00:00',
 '2025-08-31 00:00:00', '2025-09-30 00:00:00', '2025-10-31 00:00:00',
 '2025-11-30 00:00:00', '2026-01-31 00:00:00', '2026-02-28 00:00:00',
 '2023-01-31 00:00:00', '2025-12-31 00:00:00', '2026-03-31 00:00:00',
 '2026-04-30 00:00:00']
Length: 40, dtype: datetime64[ns]

In [392]:
tmp_df = tmp_df.groupby(
    ['channel_name', 'asm_area_code', 'depot_code', 
     'parent_material_code', 'month_date'], as_index=False
).sum()

In [393]:
tmp_df.duplicated(subset=['channel_name', 'asm_area_code', 'depot_code', 'parent_material_code','month_date']).sum()

0

In [394]:
actuals_df = tmp_df.copy()

del tmp_df, tmp2_df

In [395]:
actuals_df.head()

,channel_name,asm_area_code,depot_code,parent_material_code,month_date,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,B2B,BCE1,D231,702478,2024-09-30,PADV-HRCR,0.0,0.0,0.0,0.0
1,B2B,BCE1,D231,702478,2024-10-31,PADV-HRCR,0.0,0.0,0.0,0.0
2,B2B,BCE1,D231,702478,2024-12-31,PADV-HRCR,0.0,0.0,0.0,0.0
3,B2B,BCE1,D231,705148,2024-09-30,NHR-UTTAM,0.0,0.0,0.0,0.0
4,B2B,BCE1,D231,705148,2024-10-31,NHR-UTTAM,0.0,0.0,0.0,0.0


In [396]:
actuals_df['sec_actuals_vol_rum'].sum()

216496363.2890001

In [397]:
# actuals_df = actuals_df.groupby(['channel_name', 'depot_code', 'parent_material_code',
#        'material_group_code', 'month_date'])[['pri_actuals_vol_rum',
#        'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum().reset_index()
# actuals_df

In [398]:
# actuals_df[actuals_df['channel_name'] == 'GT'].to_csv('trend_GT.csv')

In [399]:
actuals_df = actuals_df.rename(columns={
    'channel_name': 'channel',
    'asm_area_code': 'asm',
    'depot_code': 'depot',
    'parent_material_code': 'psku',
    'sec_apo_plan_vol_rum': 'Consensus Vol',
    'sec_actuals_vol_rum': 'Actuals Vol'
})

In [400]:
actuals_df.columns

Index(['channel', 'asm', 'depot', 'psku', 'month_date', 'material_group_code',
       'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'Consensus Vol',
       'Actuals Vol'],
      dtype='object')

In [401]:
actuals_df = actuals_df.groupby(['channel', 'depot', 'psku', 'month_date'])[['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'Consensus Vol',
       'Actuals Vol']].sum().reset_index()

In [402]:
df = df.groupby(['channel', 'portfolio', 'brand', 'run_month', 'm month',
       'month', 'depot', 'psku'])[['pred vol (roum)', 'prophet vol',
       'rf_vol', 'prophet heuristic vol', 'rf heuristic vol']].sum().reset_index()

In [403]:
df['psku'] = df['psku'].astype(int)
actuals_df['psku'] = actuals_df['psku'].astype(int)

In [404]:
actuals_df.rename(columns = {'month_date':'month'}, inplace = True)

In [405]:
df

,channel,portfolio,brand,run_month,m month,month,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol
0,B2B,CNO,NHR-UTTAM,2025-10-31,M,2025-10-31,D231,718474,0.027000,0.000000,0.029317,0.027000,0.029317
1,B2B,CNO,NHR-UTTAM,2025-10-31,M,2025-10-31,D231,718475,0.162094,0.162094,0.061539,0.068033,0.068033
2,B2B,CNO,NHR-UTTAM,2025-10-31,M,2025-10-31,D231,718476,0.156568,0.156568,0.066100,0.024700,0.031400
3,B2B,CNO,NHR-UTTAM,2025-10-31,M,2025-10-31,D232,718475,0.000000,0.000000,0.001933,0.000000,0.000000
4,B2B,CNO,NHR-UTTAM,2025-10-31,M+1,2025-11-30,D231,718474,0.009717,0.008834,0.008817,0.009717,0.027000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
397967,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810406,0.000000,0.000000,0.000000,0.000000,0.000000
397968,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810407,0.000000,0.000000,0.000000,0.000000,0.000000
397969,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,808489,0.000000,9.892327,4.895000,0.000000,0.000000
397970,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,809750,0.000000,0.000000,0.000000,0.000000,0.000000


In [ ]:
# duplicates = actuals_df[actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date'], keep=False)]

# # Get indices of duplicates that are NOT PABABY_ML
# indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# # Remove those rows
# actuals_df = actuals_df.drop(indices_to_drop)

# # Verify no duplicates remain
# print(actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date']).sum())

In [407]:
actuals_df[actuals_df.duplicated(subset=['channel', 'depot', 'psku', 'month'], keep=False)]

,channel,depot,psku,month,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol


In [409]:
len_before_merge = len(df)
df = df.merge(
    actuals_df.drop([ 'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum'], axis=1),
    on=['channel', 'depot', 'psku', 'month'],
    how='left'
)
assert len_before_merge == len(df)
del len_before_merge

In [410]:
df

,channel,portfolio,brand,run_month,m month,month,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,Consensus Vol,Actuals Vol
0,B2B,CNO,NHR-UTTAM,2025-10-31,M,2025-10-31,D231,718474,0.027000,0.000000,0.029317,0.027000,0.029317,NaN,NaN
1,B2B,CNO,NHR-UTTAM,2025-10-31,M,2025-10-31,D231,718475,0.162094,0.162094,0.061539,0.068033,0.068033,0.0,0.156
2,B2B,CNO,NHR-UTTAM,2025-10-31,M,2025-10-31,D231,718476,0.156568,0.156568,0.066100,0.024700,0.031400,0.0,0.028
3,B2B,CNO,NHR-UTTAM,2025-10-31,M,2025-10-31,D232,718475,0.000000,0.000000,0.001933,0.000000,0.000000,NaN,NaN
4,B2B,CNO,NHR-UTTAM,2025-10-31,M+1,2025-11-30,D231,718474,0.009717,0.008834,0.008817,0.009717,0.027000,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397967,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810406,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN
397968,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810407,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN
397969,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,808489,0.000000,9.892327,4.895000,0.000000,0.000000,NaN,NaN
397970,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,809750,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN


In [411]:
df[(df['channel'] == 'GT') & (df['month'] == '2026-04-30') & (df['run_month'] == '2026-03-31')]['pred vol (roum)'].sum()

5346018.003690124

In [143]:
df[(df['channel'] == 'GT') & (df['Actuals Vol'].isna())]#['pred vol (roum)'].sum()#.isnull().sum()

,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,Consensus Vol,Actuals Vol
125942,GT,CNO,KERALA,0x2a,2025-12-31,M+2,2026-02-28,KARN,D677,718314,0.00000,0.0,0.0,0.0,0.0,NaN,NaN
125943,GT,CNO,KERALA,0x2a,2025-12-31,M+3,2026-03-31,KARN,D677,718314,0.00000,0.0,0.0,0.0,0.0,NaN,NaN
125947,GT,CNO,KERALA,A,2026-01-31,M+4,2026-05-31,KRL,D676,718314,0.00000,0.0,0.0,0.0,0.0,NaN,NaN
125948,GT,CNO,KERALA,A,2026-01-31,M+5,2026-06-30,KRL,D676,718314,0.00000,0.0,0.0,0.0,0.0,NaN,NaN
125971,GT,CNO,KERALA,B,2025-10-31,M+1,2025-11-30,KARN,D677,718314,0.00000,0.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
330257,GT,Skin Care,PAD_BDYOL,0x2a,2026-03-31,M+4,2026-07-31,ORS,D535,719116,3.62399,0.0,0.0,0.0,0.0,NaN,NaN
330258,GT,Skin Care,PAD_BDYOL,0x2a,2026-03-31,M+4,2026-07-31,RWBN,D231,719116,0.00000,0.0,0.0,0.0,0.0,NaN,NaN
330259,GT,Skin Care,PAD_BDYOL,0x2a,2026-03-31,M+4,2026-07-31,RWBN,D231,719117,0.00000,0.0,0.0,0.0,0.0,NaN,NaN
330260,GT,Skin Care,PAD_BDYOL,0x2a,2026-03-31,M+4,2026-07-31,RWBN,D231,719118,0.00000,0.0,0.0,0.0,0.0,NaN,NaN


In [427]:
# actuals_df[(actuals_df['asm'] == 'KARN') & (actuals_df['psku'] == 718314)]
actuals_df['key'] =  actuals_df['depot'] + actuals_df['psku'].astype(str)
df['key'] = df['depot'] + df['psku'].astype(str)


In [428]:
new_df = actuals_df[(actuals_df['key'].isin(df['key'].unique()))]
new_df

,channel,depot,psku,month,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol,key
191,B2B,BUDZ,733006,2026-02-28,0.124,0.000000,0.0,0.124,BUDZ733006
192,B2B,BUDZ,733006,2026-03-31,0.200,0.000000,0.0,0.200,BUDZ733006
193,B2B,BUDZ,733006,2026-04-30,1.900,0.000000,0.0,1.900,BUDZ733006
194,B2B,BUDZ,734542,2026-04-30,1.216,0.000000,0.0,1.216,BUDZ734542
195,B2B,BUDZ,734552,2026-02-28,0.400,0.000000,0.0,0.400,BUDZ734552
...,...,...,...,...,...,...,...,...,...
1583453,QCOM,D677,810738,2025-11-30,0.000,0.000000,0.0,0.000,D677810738
1583454,QCOM,D677,810738,2025-12-31,0.000,2.991578,0.0,0.000,D677810738
1583455,QCOM,D677,810738,2026-01-31,17.376,0.000000,0.0,17.376,D677810738
1583456,QCOM,D677,810738,2026-02-28,8.688,0.000000,0.0,8.688,D677810738


In [429]:
new_df = new_df.merge(df[['key','brand']].drop_duplicates(), on = 'key', how = 'left')

In [430]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()

len_before_merge = len(new_df)
new_df = new_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(new_df)

new_df['Consensus Vol'] = new_df['Consensus Vol'].fillna(0)
new_df['Actuals Vol'] = new_df['Actuals Vol'].fillna(0)

new_df['Consensus Val'] = new_df['Consensus Vol'] * new_df['Index Rate'] / (10 ** 7)
new_df['Actuals Val'] = new_df['Actuals Vol'] * new_df['Index Rate'] / (10 ** 7)

In [431]:
new_df[(new_df['channel'] == 'GT') & (new_df['month'] == '2026-04-30')]['Actuals Val'].sum()

494.91703558113693

In [432]:
new_df[(new_df['channel'] == 'GT')].to_csv('trend_gt4.csv')

In [181]:
actuals_df[(actuals_df['key'].isin(df['key'].unique())) & (actuals_df['channel'] == 'GT')].to_csv('trend_gt2.csv')

In [185]:
actuals_df[(actuals_df['key'].isin(df['key'].unique())) & (actuals_df['channel'] == 'GT') & (actuals_df['month'] == '2026-04-30')]['Actuals Vol'].sum()

5101623.744999999

In [151]:
extra_key = actuals_df[~actuals_df['key'].isin(df['key'].unique())]
extra_key

,channel,asm,depot,psku,month,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol,key
0,B2B,BCE1,D231,702478,2024-09-30,PADV-HRCR,0.000,0.000000,0.0000,0.000,BCE1D231702478
1,B2B,BCE1,D231,702478,2024-10-31,PADV-HRCR,0.000,0.000000,0.0000,0.000,BCE1D231702478
2,B2B,BCE1,D231,702478,2024-12-31,PADV-HRCR,0.000,0.000000,0.0000,0.000,BCE1D231702478
3,B2B,BCE1,D231,705148,2024-09-30,NHR-UTTAM,0.000,0.000000,0.0000,0.000,BCE1D231705148
4,B2B,BCE1,D231,705148,2024-10-31,NHR-UTTAM,0.000,0.000000,0.0000,0.000,BCE1D231705148
...,...,...,...,...,...,...,...,...,...,...,...
2189256,QCOM,QCW2,D463,811021,2026-04-30,PADV_WIPS,0.000,17.993646,0.0000,0.000,QCW2D463811021
2189257,QCOM,QCW2,D463,811169,2026-03-31,SW_SGPRF,0.000,13.043478,0.0000,0.000,QCW2D463811169
2189258,QCOM,QCW2,D463,811169,2026-04-30,SW_SGPRF,0.000,3.969082,0.0000,0.000,QCW2D463811169
2189259,QCOM,QCW2,D463,811181,2026-04-30,SAF_CDPRS,0.096,0.000000,0.0000,0.096,QCW2D463811181


In [177]:
extra_key[(extra_key['channel'] == 'GT') & (extra_key['month'] == '2026-04-30')]['Actuals Vol'].sum()

1467974.5410000002

In [175]:
extra_key[(extra_key['channel'] == 'GT') ]

,channel,asm,depot,psku,month,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol,key
532468,GT,AURG,D3A4,702478,2023-02-28,PADV-HRCR,0.0,0.0,0.0,0.0,AURGD3A4702478
532469,GT,AURG,D3A4,702478,2023-03-31,PADV-HRCR,0.0,0.0,0.0,0.0,AURGD3A4702478
532470,GT,AURG,D3A4,702478,2023-04-30,PADV-HRCR,0.0,0.0,0.0,0.0,AURGD3A4702478
532471,GT,AURG,D3A4,702478,2023-05-31,PADV-HRCR,0.0,0.0,0.0,0.0,AURGD3A4702478
532472,GT,AURG,D3A4,702930,2023-02-28,PADV-HRGL,0.0,0.0,0.0,0.0,AURGD3A4702930
...,...,...,...,...,...,...,...,...,...,...,...
1701284,GT,WUP,D113,810439,2025-04-30,SAF-MUSLI,0.0,0.0,0.0,0.0,WUPD113810439
1701285,GT,WUP,D113,811149,2026-01-31,SW_SGPRF,0.0,0.0,0.0,0.0,WUPD113811149
1701286,GT,WUP,D113,811150,2026-01-31,SW_SGPRF,0.0,0.0,0.0,0.0,WUPD113811150
1701287,GT,WUP,D113,811151,2026-01-31,SW_SGPRF,0.0,0.0,0.0,0.0,WUPD113811151


In [412]:
df['channel'].unique()

array(['B2B', 'ECOM', 'GT', 'MT', 'QCOM'], dtype=object)

In [44]:
df

,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,Consensus Vol,Actuals Vol
0,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D231,718474,0.027000,0.000000,0.029317,0.027000,0.029317,NaN,NaN
1,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D231,718475,0.162094,0.162094,0.061539,0.068033,0.068033,NaN,NaN
2,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D231,718476,0.156568,0.156568,0.066100,0.024700,0.031400,NaN,NaN
3,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D232,718475,0.000000,0.000000,0.001933,0.000000,0.000000,NaN,NaN
4,B2B,CNO,NHR-UTTAM,B,2025-10-31,M+1,2025-11-30,BCE1,D231,718474,0.009717,0.008834,0.008817,0.009717,0.027000,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501296,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809807,0.000000,14.685752,2.232000,0.000000,0.000000,NaN,NaN
501297,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809850,0.000000,0.000000,0.120000,0.000000,0.000000,NaN,NaN
501298,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809865,0.000000,0.000000,0.746667,0.000000,0.000000,NaN,NaN
501299,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809866,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN


In [45]:
df#.isnull().sum()

,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,Consensus Vol,Actuals Vol
0,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D231,718474,0.027000,0.000000,0.029317,0.027000,0.029317,NaN,NaN
1,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D231,718475,0.162094,0.162094,0.061539,0.068033,0.068033,NaN,NaN
2,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D231,718476,0.156568,0.156568,0.066100,0.024700,0.031400,NaN,NaN
3,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D232,718475,0.000000,0.000000,0.001933,0.000000,0.000000,NaN,NaN
4,B2B,CNO,NHR-UTTAM,B,2025-10-31,M+1,2025-11-30,BCE1,D231,718474,0.009717,0.008834,0.008817,0.009717,0.027000,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501296,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809807,0.000000,14.685752,2.232000,0.000000,0.000000,NaN,NaN
501297,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809850,0.000000,0.000000,0.120000,0.000000,0.000000,NaN,NaN
501298,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809865,0.000000,0.000000,0.746667,0.000000,0.000000,NaN,NaN
501299,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809866,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN


In [413]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,TRU_PDRFR,850.57000
1,2027-03-31,TRU_OATS,177.07000
2,2027-03-31,TRU_QUINO,204.75000
3,2027-03-31,TRU_RAW,453.44000
4,2027-03-31,NHR_NHO_E,266.57953


In [414]:
qtr_ind_rate_df[qtr_ind_rate_df['brand_code'] == 'PCNO(R)']

,month_date,brand_code,qtr_ind_rate
68,2027-03-31,PCNO(R),349274.00142


In [415]:
len_before_merge = len(df)
df = df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(df)
del len_before_merge

In [416]:
df['brand'].unique()

array(['NHR-UTTAM', 'PCNO FLEX', 'PCNO(R)', 'CO_SO_VCN', 'SAF-MUSLI',
       'SAFF OATS', 'SAFF SALT', 'SAFF_ODLS', 'SAF_HONEY', 'SAF_MAYO',
       'SAF_MILET', 'SAF_PNBTR', 'SFOAT-CUP', 'SFOATS-FL', 'SFOATS_MG',
       'SF_IM_CHY', 'SF_MNCHPS', 'SF_SOYACN', 'ADV-AHO-R', 'H&C',
       'H&C_ALMND', 'NHR NSJ H', 'NHR-SABDM', 'NHR_ALOAM', 'NHR_SSAHO',
       'NIHAR NHO', 'PA-ALO-HO', 'PADV-HOT', 'PADVJAS-R', 'PA_AMVITE',
       'PA_CN_HO', 'PA_EXT_ML', 'PA_JASGLD', 'P_AL_GOLD', 'P_EN_ALM',
       'P_EN_BGHB', 'P_EN_CRSH', 'P_EN_RSMR', 'BRD_BDOIL', 'BRD_BDSPR',
       'BRD_DOGAS', 'BRD_FSWSH', 'BRD_HROIL', 'BRD_HRWAX', 'BRD_PERFM',
       'PADV-HRCR', 'SW HRGEL', 'SW HSPRY', 'SW NOGAS', 'SW STLDEO',
       'SW_HR_WAX', 'NHR_VTEHO', 'LVN_SHMP', 'MALO-NATU', 'MALT-NATU',
       'REV.LQDST', 'REV.ST.', 'REV_LQFRG', 'HC SNS', 'LIVON',
       'LIVON S-R', 'LVN_SR_DR', 'SAFF ACTV', 'SAFF GOLD', 'SAFF KO',
       'SAFF KOCO', 'BIO OILS', 'PA-BDYLOT', 'SFOATS_GD', 'H&C DFOIL',
       'HC_PBHOIL', 

In [164]:
df

,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,Consensus Vol,Actuals Vol,key,Index Rate
0,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D231,718474,0.027000,0.000000,0.029317,0.027000,0.029317,NaN,NaN,BCE1D231718474,250000.000000
1,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D231,718475,0.162094,0.162094,0.061539,0.068033,0.068033,0.0,0.156,BCE1D231718475,250000.000000
2,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D231,718476,0.156568,0.156568,0.066100,0.024700,0.031400,0.0,0.028,BCE1D231718476,250000.000000
3,B2B,CNO,NHR-UTTAM,B,2025-10-31,M,2025-10-31,BCE1,D232,718475,0.000000,0.000000,0.001933,0.000000,0.000000,NaN,NaN,BCE1D232718475,250000.000000
4,B2B,CNO,NHR-UTTAM,B,2025-10-31,M+1,2025-11-30,BCE1,D231,718474,0.009717,0.008834,0.008817,0.009717,0.027000,NaN,NaN,BCE1D231718474,250000.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501296,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809807,0.000000,14.685752,2.232000,0.000000,0.000000,NaN,NaN,QCW2D461809807,1779.273746
501297,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809850,0.000000,0.000000,0.120000,0.000000,0.000000,NaN,NaN,QCW2D461809850,1779.273746
501298,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809865,0.000000,0.000000,0.746667,0.000000,0.000000,NaN,NaN,QCW2D461809865,1779.273746
501299,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809866,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,QCW2D461809866,1779.273746


In [417]:
df.isna().sum()

channel                       0
portfolio                     0
brand                         0
run_month                     0
m month                       0
month                         0
depot                         0
psku                          0
pred vol (roum)               0
prophet vol                   0
rf_vol                        0
prophet heuristic vol         0
rf heuristic vol              0
Consensus Vol            137355
Actuals Vol              137355
Index Rate                   12
dtype: int64

In [418]:
df['Consensus Vol'] = df['Consensus Vol'].fillna(0)
df['Actuals Vol'] = df['Actuals Vol'].fillna(0)

In [419]:
df.columns

Index(['channel', 'portfolio', 'brand', 'run_month', 'm month', 'month',
       'depot', 'psku', 'pred vol (roum)', 'prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol', 'Consensus Vol',
       'Actuals Vol', 'Index Rate'],
      dtype='object')

In [420]:
df.rename(columns={'pred vol (roum)': 'Stat Vol'}, inplace=True)

In [421]:
df['Stat Val'] = df['Stat Vol'] * df['Index Rate'] / (10 ** 7)
df['Consensus Val'] = df['Consensus Vol'] * df['Index Rate'] / (10 ** 7)
df['Actuals Val'] = df['Actuals Vol'] * df['Index Rate'] / (10 ** 7)

In [174]:
df[(df['channel'] == 'GT') & (df['Actuals Vol'].isna()) & (df['run_month'] == '2026-03-31') & (df['m month'] == 'M+1')]['Stat Val'].sum()

0.0

In [426]:
df[(df['run_month'] == '2026-03-31') & (df['m month'] == 'M+1') & (df['channel'] == 'GT')]['Stat Val'].sum()

449.7762030932171

In [433]:
df['Stat Error'] = df['Stat Val'] - df['Actuals Val']
df['Consensus Error'] = df['Consensus Val'] - df['Actuals Val']

df['Stat Abs Error'] = np.abs(df['Stat Error'])
df['Consensus Abs Error'] = np.abs(df['Consensus Error'])

In [434]:
df.columns

Index(['channel', 'portfolio', 'brand', 'run_month', 'm month', 'month',
       'depot', 'psku', 'Stat Vol', 'prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol', 'Consensus Vol',
       'Actuals Vol', 'Index Rate', 'Stat Val', 'Consensus Val', 'Actuals Val',
       'key', 'Stat Error', 'Consensus Error', 'Stat Abs Error',
       'Consensus Abs Error'],
      dtype='object')

In [435]:
for col in ['prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol']:
    df[f'{col}_value'] = df[col] * df['Index Rate'] / (10 ** 7)

In [436]:
df['stat_bias'] = df['Stat Error']/df['Actuals Val']

In [437]:
df = df.fillna(0)

In [438]:
df = df[df['m month'] == 'M+1']#.isnull().sum()

In [439]:
import numpy as np
import pandas as pd

df['stat_bias'] = (
    df['stat_bias']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
labels = [
    '< -15%',
    '-15% to -10%',
    '-10% to -5%',
    '-5% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 15%',
    '> 15%'
]

df['stat_bias_bucket'] = pd.cut(
    df['stat_bias'],
    bins=bins,
    labels=labels,
    right=False   # 👈 key change
)


In [440]:
df

,channel,portfolio,brand,run_month,m month,month,depot,psku,Stat Vol,prophet vol,...,Stat Error,Consensus Error,Stat Abs Error,Consensus Abs Error,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value,stat_bias,stat_bias_bucket
4,B2B,CNO,NHR-UTTAM,2025-10-31,M+1,2025-11-30,D231,718474,0.009717,0.008834,...,0.000243,0.000000,0.000243,0.000000,0.000221,0.000220,0.000243,0.000675,0.000000,0% to 5%
5,B2B,CNO,NHR-UTTAM,2025-10-31,M+1,2025-11-30,D231,718475,0.001466,0.001333,...,-0.000838,-0.000875,0.000838,0.000875,0.000033,0.000723,0.000037,0.001701,-0.958118,< -15%
6,B2B,CNO,NHR-UTTAM,2025-10-31,M+1,2025-11-30,D231,718476,0.083071,0.114477,...,0.001727,-0.000350,0.001727,0.000350,0.002862,0.002022,0.000617,0.000950,4.933610,> 15%
7,B2B,CNO,NHR-UTTAM,2025-10-31,M+1,2025-11-30,D232,718475,0.042974,0.042974,...,0.001074,0.000000,0.001074,0.000000,0.001074,0.000435,0.000000,0.000000,0.000000,0% to 5%
28,B2B,CNO,NHR-UTTAM,2025-11-30,M+1,2025-12-31,D231,718474,0.001261,0.001261,...,0.000032,0.000000,0.000032,0.000000,0.000032,0.000179,0.000675,0.000675,0.000000,0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397553,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+1,2025-12-31,D674,810406,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
397554,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+1,2025-12-31,D674,810407,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
397555,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+1,2025-12-31,D676,808489,0.000000,23.308381,...,0.000000,0.000000,0.000000,0.000000,0.004147,0.001373,0.000000,0.000000,0.000000,0% to 5%
397556,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+1,2025-12-31,D676,809750,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%


In [441]:
df['forecast_granularity'] = 'Depot x PSKU'
df['forecast_type'] = 'secondary'

In [66]:
cols = ['forecast_granularity', 'forecast_type'] + [
    c for c in df.columns
    if c not in ['forecast_granularity', 'forecast_type']
]

df = df[cols]

In [67]:
# df.to_csv('accuracy_framework.csv')

In [68]:
# delivery_df = pd.read_csv('/data/aman_singh/acuuracy_check/Export View of Month Vol & APO.csv')

# delivery_df['Channel'] = delivery_df['Channel'].replace({
#     'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
# delivery_df

In [69]:
# delivery_df['delivery_vol'] = delivery_df['01-12-2025 Del Vol'].map(lambda x:0 if x.strip() == '-' else float(x.strip().replace(',','')))

In [70]:
# delivery_df.to_csv('delivery_data.csv')

In [71]:
# delivery_df = delivery_df.groupby(['Channel', 'Depot','ASM', 'PSKU'])['delivery_vol'].sum().reset_index(
# )#.rename(columns = {'01-12-2025 Del Vol':'delivery_vol'})
# delivery_df

In [72]:
# delivery_df.columns = delivery_df.columns.str.lower()

In [73]:
# delivery_df['month'] = pd.to_datetime('2025-12-31')

In [74]:
# len_before_merge = len(df)
# df = df.merge(
#     delivery_df,
#     on=['channel', 'asm', 'depot', 'psku', 'month'],
#     how='left'
# )
# assert len_before_merge == len(df)
# del len_before_merge

In [75]:
# df['Delivery Val'] = df['delivery_vol'] * df['Index Rate'] / (10 ** 7)

# df['Dp Error'] = df['Delivery Val'] - df['Actuals Val']
# # df['Consensus Error_del'] = df['Consensus Val'] - df['Delivery Val']

# df['Dp Abs Error'] = np.abs(df['Dp Error'])
# # df['Consensus Abs Error_del'] = np.abs(df['Consensus Error_del'])

In [76]:
# df.to_excel("accuracy_framework2.xlsx", index=False)

In [77]:
# df = pd.read_csv('accuracy_framework.csv')

### Qcom

### Helper Functions

In [5]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

# realignment_df = realignment_df[
#     realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column, channel='QCOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        realignment_data['channel'].isin([channel, channel + ' B2C', 'ALL'])
    ]

    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

### Forecast

In [6]:
qcom_df = pd.read_sql(
    """select * from TRN_MIL_DF_OFT2PRIM_SHARED""",
    dev_conn
)
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month'] = pd.to_datetime(qcom_df['month'])

In [7]:
qcom_df#[qcom_df['Channel']=='Ecom']

,month,depot,psku,brand,portfolio,m month,run_month,calculated primary vol,channel
0,2025-11-30,0x2a,718287,PCNO(R),CNO,M,2025-11-30,0.0,Qcom
1,2025-11-30,0x2a,718288,SAFF GOLD,Saffola Oils,M,2025-11-30,0.0,Qcom
2,2025-11-30,0x2a,718297,PCNO(R),CNO,M,2025-11-30,0.0,Qcom
3,2025-11-30,0x2a,718299,PCNO(R),CNO,M,2025-11-30,0.0,Qcom
4,2025-11-30,0x2a,718300,PCNO FLEX,CNO,M,2025-11-30,0.0,Qcom
...,...,...,...,...,...,...,...,...,...
349513,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom
349514,2026-07-31,D677,810807,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom
349515,2026-07-31,D677,810919,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom
349516,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.0,Ecom


### Actuals

In [8]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

# realignment_df = realignment_df[
#     realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column, channel='QCOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        realignment_data['channel'].isin([channel, channel + ' B2C', 'ALL'])
    ]

    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [10]:
depot_psku_primary_query = """
SELECT
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Grofers', 'Zepto', 'Kiranakart Technologies', 'Swiggy', 'ZEPTO')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
where month_date BETWEEN '2023-01-01' AND '2026-04-30' 
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    depot_psku_primary_query,
    prod_conn
)
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)
depot_psku_primary_df['month_date'] = pd.to_datetime(depot_psku_primary_df['month_date'])
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code')
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['depot_code', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]
depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date']).sum()
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [11]:
depot_psku_primary_df['channel'] = 'Qcom'
depot_psku_primary_df_qcom = depot_psku_primary_df.copy()

In [13]:
depot_psku_primary_df_qcom.to_csv('Trend_qcom.csv')

In [448]:
depot_psku_primary_query = """
SELECT
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Flipkart-National', 'Flipkart-Grocery', 'Big basket B2C', 'RK WORLDINFOCOM', 'Amazon B2C', 
        'FATEHPURIA HYGIENE', 'Nykaa', 'Flipkart-Minutes', 'Purplle', 'Myntra', 'Dealshare', 'FlipkartGrocery', 'City Mall', '1MG', 
        'ARIPL', 'First Cry', 'Meesho', 'RKWorld', 'CITIMALL', 'Firstcry', 'EMAZING DEALS', 'MYNTRA')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
where month_date BETWEEN '2025-12-01' AND '2026-04-30' 
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    depot_psku_primary_query,
    prod_conn
)
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)
depot_psku_primary_df['month_date'] = pd.to_datetime(depot_psku_primary_df['month_date'])
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code', channel='ECOM')
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['depot_code', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]
depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date']).sum()
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [449]:
depot_psku_primary_df['channel'] = 'Ecom'
depot_psku_primary_df_ecom = depot_psku_primary_df.copy()

In [450]:
depot_psku_primary_df = pd.concat([depot_psku_primary_df_qcom, depot_psku_primary_df_ecom], ignore_index=True)
depot_psku_primary_df

,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,channel
0,D112,718299,PCNO(R),2026-03-31,0.019,0.021815,0.0000,0.019,Qcom
1,D112,718299,PCNO(R),2026-04-30,0.076,0.024718,0.0000,0.076,Qcom
2,D112,718310,PCNO(R),2026-01-31,0.000,0.012857,0.0000,0.000,Qcom
3,D112,718310,PCNO(R),2026-03-31,0.000,0.000266,0.0000,0.000,Qcom
4,D112,718312,PCNO(R),2025-12-31,0.460,0.958689,1.1501,0.460,Qcom
...,...,...,...,...,...,...,...,...,...
46289,D677,811181,SAF_CDPRS,2026-03-31,0.000,0.046656,0.0000,0.000,Ecom
46290,D677,811181,SAF_CDPRS,2026-04-30,0.000,0.020770,0.0000,0.000,Ecom
46291,D677,811267,PA_ESS_HO,2026-04-30,0.000,0.574270,0.0000,0.000,Ecom
46292,D677,811268,PA_ESS_HO,2026-04-30,0.000,0.368192,0.0000,0.000,Ecom


In [451]:
actuals_df = depot_psku_primary_df.copy()

In [452]:
actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date']).sum()

94

In [453]:
# For duplicates, keep only the row with PABABY_ML material_group_code
duplicates = actuals_df[actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date'], keep=False)]

# Get indices of duplicates that are NOT PABABY_ML
indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# Remove those rows
actuals_df = actuals_df.drop(indices_to_drop)

# Verify no duplicates remain
print(actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date']).sum())

0


In [454]:
actuals_df = actuals_df.groupby(['channel','depot_code', 'parent_material_code',
       'material_group_code', 'month_date'])[['pri_actuals_vol_rum',
       'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum().reset_index()
actuals_df

,channel,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,Ecom,D112,718287,PCNO(R),2025-12-31,0.000,0.023463,0.0000,0.000
1,Ecom,D112,718288,SAFF GOLD,2026-03-31,0.000,0.000000,1.9086,0.000
2,Ecom,D112,718299,PCNO(R),2026-04-30,0.000,0.038390,0.0618,0.000
3,Ecom,D112,718308,PCNO(R),2025-12-31,0.126,0.000000,0.0000,0.126
4,Ecom,D112,718308,PCNO(R),2026-01-31,0.036,0.000000,0.0000,0.036
...,...,...,...,...,...,...,...,...,...
46195,Qcom,D677,811169,SW_SGPRF,2026-04-30,0.000,1.417058,0.0000,0.000
46196,Qcom,D677,811181,SAF_CDPRS,2026-02-28,0.000,0.399990,0.0000,0.000
46197,Qcom,D677,811181,SAF_CDPRS,2026-03-31,0.000,0.266666,0.0000,0.000
46198,Qcom,D677,811181,SAF_CDPRS,2026-04-30,0.036,0.121976,0.0000,0.036


In [455]:
actuals_df.columns

Index(['channel', 'depot_code', 'parent_material_code', 'material_group_code',
       'month_date', 'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum',
       'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum'],
      dtype='object')

In [456]:
actuals_df = actuals_df.rename(columns={
    'depot_code': 'depot',
    'parent_material_code': 'psku',
    'sec_apo_plan_vol_rum': 'Consensus Vol',
    'sec_actuals_vol_rum': 'Actuals Vol',
    'month_date': 'month'
})

In [457]:
actuals_df.head()

,channel,depot,psku,material_group_code,month,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol
0,Ecom,D112,718287,PCNO(R),2025-12-31,0.000,0.023463,0.0000,0.000
1,Ecom,D112,718288,SAFF GOLD,2026-03-31,0.000,0.000000,1.9086,0.000
2,Ecom,D112,718299,PCNO(R),2026-04-30,0.000,0.038390,0.0618,0.000
3,Ecom,D112,718308,PCNO(R),2025-12-31,0.126,0.000000,0.0000,0.126
4,Ecom,D112,718308,PCNO(R),2026-01-31,0.036,0.000000,0.0000,0.036


In [94]:
#qcom_df.rename(columns = {'Channel':'channel'}, inplace = True)

In [458]:
qcom_df['psku'] = qcom_df['psku'].astype(int)
actuals_df['psku'] = actuals_df['psku'].astype(int)

In [96]:
# actuals_df = actuals_df[actuals_df['month_date'] == '2025-12-31']
# actuals_df

In [459]:
len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    actuals_df.drop([ 'material_group_code', 'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum'], axis=1),
    on=['channel','depot', 'psku', 'month'],
    how='left'
)
assert len_before_merge == len(qcom_df)
del len_before_merge

In [460]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,TRU_PDRFR,850.57000
1,2027-03-31,TRU_OATS,177.07000
2,2027-03-31,TRU_QUINO,204.75000
3,2027-03-31,TRU_RAW,453.44000
4,2027-03-31,NHR_NHO_E,266.57953


In [461]:
len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(qcom_df)
del len_before_merge

In [462]:
qcom_df.isna().sum()

month                          0
depot                          0
psku                           0
brand                          0
portfolio                      0
m month                        0
run_month                      0
calculated primary vol         0
channel                        0
Consensus Vol             270298
Actuals Vol               270298
Index Rate                     0
dtype: int64

In [463]:
qcom_df['Consensus Vol'] = qcom_df['Consensus Vol'].fillna(0)
qcom_df['Actuals Vol'] = qcom_df['Actuals Vol'].fillna(0)

In [464]:
qcom_df['Consensus Vol'].max()

16869.7071

In [465]:
qcom_df.rename(columns={'calculated primary vol': 'Stat Vol'}, inplace=True)
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate
0,2025-11-30,0x2a,718287,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,349274.001420
1,2025-11-30,0x2a,718288,SAFF GOLD,Saffola Oils,M,2025-11-30,0.0,Qcom,0.0,0.0,138865.260689
2,2025-11-30,0x2a,718297,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,349274.001420
3,2025-11-30,0x2a,718299,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,349274.001420
4,2025-11-30,0x2a,718300,PCNO FLEX,CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,230000.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
349513,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom,0.0,0.0,366.484998
349514,2026-07-31,D677,810807,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom,0.0,0.0,366.484998
349515,2026-07-31,D677,810919,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom,0.0,0.0,366.484998
349516,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.0,Ecom,0.0,0.0,1712.605337


In [466]:
qcom_df['Stat Val'] = qcom_df['Stat Vol'] * qcom_df['Index Rate'] / (10 ** 7)
qcom_df['Consensus Val'] = qcom_df['Consensus Vol'] * qcom_df['Index Rate'] / (10 ** 7)
qcom_df['Actuals Val'] = qcom_df['Actuals Vol'] * qcom_df['Index Rate'] / (10 ** 7)

In [467]:
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate,Stat Val,Consensus Val,Actuals Val
0,2025-11-30,0x2a,718287,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,349274.001420,0.0,0.0,0.0
1,2025-11-30,0x2a,718288,SAFF GOLD,Saffola Oils,M,2025-11-30,0.0,Qcom,0.0,0.0,138865.260689,0.0,0.0,0.0
2,2025-11-30,0x2a,718297,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,349274.001420,0.0,0.0,0.0
3,2025-11-30,0x2a,718299,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,349274.001420,0.0,0.0,0.0
4,2025-11-30,0x2a,718300,PCNO FLEX,CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,230000.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349513,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom,0.0,0.0,366.484998,0.0,0.0,0.0
349514,2026-07-31,D677,810807,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom,0.0,0.0,366.484998,0.0,0.0,0.0
349515,2026-07-31,D677,810919,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom,0.0,0.0,366.484998,0.0,0.0,0.0
349516,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.0,Ecom,0.0,0.0,1712.605337,0.0,0.0,0.0


In [468]:
qcom_df[(qcom_df['month'] == '2026-04-30') & (qcom_df['channel']=='Ecom') & (qcom_df['run_month']=='2026-03-31')]['Actuals Val'].sum()

37.233712446657215

In [469]:
qcom_df['Stat Error'] = qcom_df['Stat Val'] - qcom_df['Actuals Val']
qcom_df['Consensus Error'] = qcom_df['Consensus Val'] - qcom_df['Actuals Val']

qcom_df['Stat Abs Error'] = np.abs(qcom_df['Stat Error'])
qcom_df['Consensus Abs Error'] = np.abs(qcom_df['Consensus Error'])

In [470]:
qcom_df['stat_bias'] = qcom_df['Stat Error']/qcom_df['Actuals Val']
qcom_df = qcom_df.fillna(0)

import numpy as np
import pandas as pd

qcom_df['stat_bias'] = (
    qcom_df['stat_bias']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
labels = [
    '< -15%',
    '-15% to -10%',
    '-10% to -5%',
    '-5% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 15%',
    '> 15%'
]

qcom_df['stat_bias_bucket'] = pd.cut(
    qcom_df['stat_bias'],
    bins=bins,
    labels=labels,
    right=False  
)


In [109]:
# delivery_df = pd.read_csv('/data/aman_singh/acuuracy_check/Export View of Month Vol & APO.csv')

# delivery_df['Channel'] = delivery_df['Channel'].replace({
#     'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
# delivery_df
# delivery_df['delivery_vol'] = delivery_df['01-12-2025 Del Vol'].map(lambda x:0 if x.strip() == '-' else float(x.strip().replace(',','')))
# delivery_df = delivery_df[delivery_df['Channel'] == 'QCOM']
# delivery_df = delivery_df.groupby(['Depot','PSKU'])['delivery_vol'].sum().reset_index(
# )#.rename(columns = {'01-12-2025 Del Vol':'delivery_vol'})

# len_before_merge = len(qcom_df)
# qcom_df = qcom_df.merge(
#     delivery_df,
#     on=['Depot', 'PSKU'],
#     how='left'
# )
# assert len_before_merge == len(qcom_df)
# del len_before_merge

# qcom_df['Delivery Val'] = qcom_df['delivery_vol'] * qcom_df['Index Rate'] / (10 ** 7)

# qcom_df['Dp Error'] = qcom_df['Delivery Val'] - qcom_df['Actuals Val']

# qcom_df['Dp Abs Error'] = np.abs(qcom_df['Dp Error'])



In [471]:
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,stat vol,channel,consensus vol,...,index rate,stat val,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket
0,2025-11-30,0x2a,718287,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,...,349274.001420,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
1,2025-11-30,0x2a,718288,SAFF GOLD,Saffola Oils,M,2025-11-30,0.0,Qcom,0.0,...,138865.260689,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
2,2025-11-30,0x2a,718297,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,...,349274.001420,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
3,2025-11-30,0x2a,718299,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,...,349274.001420,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
4,2025-11-30,0x2a,718300,PCNO FLEX,CNO,M,2025-11-30,0.0,Qcom,0.0,...,230000.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349513,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom,0.0,...,366.484998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
349514,2026-07-31,D677,810807,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom,0.0,...,366.484998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
349515,2026-07-31,D677,810919,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom,0.0,...,366.484998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
349516,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.0,Ecom,0.0,...,1712.605337,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%


In [111]:
# df = pd.read_excel("/data/aman_singh/acuuracy_check/acc_framework_feb.xlsx")
# df.columns = df.columns.str.lower()
# df = df[df['forecast_granularity']!='Depot x PSKU']

In [472]:
df.columns = df.columns.str.lower()

In [473]:
qcom_df.columns

Index(['month', 'depot', 'psku', 'brand', 'portfolio', 'm month', 'run_month',
       'stat vol', 'channel', 'consensus vol', 'actuals vol', 'index rate',
       'stat val', 'consensus val', 'actuals val', 'stat error',
       'consensus error', 'stat abs error', 'consensus abs error', 'stat_bias',
       'stat_bias_bucket'],
      dtype='object')

In [474]:

qcom_df['forecast_granularity'] = 'Depot x PSKU'
qcom_df['forecast_type'] = 'offtakes_to_primary'
#qcom_df['channel'] = 'QCOM'
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,stat vol,channel,consensus vol,...,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket,forecast_granularity,forecast_type
0,2025-11-30,0x2a,718287,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
1,2025-11-30,0x2a,718288,SAFF GOLD,Saffola Oils,M,2025-11-30,0.0,Qcom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
2,2025-11-30,0x2a,718297,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
3,2025-11-30,0x2a,718299,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
4,2025-11-30,0x2a,718300,PCNO FLEX,CNO,M,2025-11-30,0.0,Qcom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349513,2026-07-31,D677,810805,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
349514,2026-07-31,D677,810807,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
349515,2026-07-31,D677,810919,PABABY_GM,Skin Care,M+4,2026-03-31,0.0,Ecom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
349516,2026-07-31,D677,811169,SW_SGPRF,Male Grooming,M+4,2026-03-31,0.0,Ecom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary


In [475]:
qcom_df = qcom_df[qcom_df['m month'] == 'M+1']

In [116]:
df2 = df.groupby(['channel', 'portfolio',
       'brand', 'brand class', 'run_month', 'm month', 'month', 'depot',
       'psku'])[['consensus vol', 'actuals vol', 'stat vol',
       'stat val', 'consensus val', 'actuals val']].sum().reset_index()

In [127]:
df2[(df2['month'] == '2026-04-30') & (df2['channel']=='GT') & (df2['run_month']=='2026-03-31')]['actuals vol'].sum()
# df2['actuals vol'].sum()

5101614.17

In [117]:
df2['stat error'] = df2['stat val'] - df2['actuals val']
df2['consensus error'] = df2['consensus val'] - df2['actuals val']

df2['stat abs error'] = np.abs(df2['stat error'])
df2['consensus abs error'] = np.abs(df2['consensus error'])

In [118]:
df2[(df2['month'] == '2026-04-30') & (df2['channel']=='GT') & (df2['run_month']=='2026-03-31')]['stat val'].sum()

449.7762030932171

In [121]:
df2['forecast_granularity'] = 'Depot x PSKU'
df2['forecast_type'] = 'secondary'

In [119]:
df2 = df2[df2['m month']=='M+1']

In [476]:
final_df = pd.concat([df,qcom_df])
final_df

,channel,portfolio,brand,run_month,m month,month,depot,psku,stat vol,prophet vol,...,stat abs error,consensus abs error,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value,stat_bias,stat_bias_bucket,forecast_granularity,forecast_type
4,B2B,CNO,NHR-UTTAM,2025-10-31,M+1,2025-11-30,D231,718474,0.009717,0.008834,...,0.000243,0.000000,0.000221,0.000220,0.000243,0.000675,0.000000,0% to 5%,Depot x PSKU,secondary
5,B2B,CNO,NHR-UTTAM,2025-10-31,M+1,2025-11-30,D231,718475,0.001466,0.001333,...,0.000838,0.000875,0.000033,0.000723,0.000037,0.001701,-0.958118,< -15%,Depot x PSKU,secondary
6,B2B,CNO,NHR-UTTAM,2025-10-31,M+1,2025-11-30,D231,718476,0.083071,0.114477,...,0.001727,0.000350,0.002862,0.002022,0.000617,0.000950,4.933610,> 15%,Depot x PSKU,secondary
7,B2B,CNO,NHR-UTTAM,2025-10-31,M+1,2025-11-30,D232,718475,0.042974,0.042974,...,0.001074,0.000000,0.001074,0.000435,0.000000,0.000000,0.000000,0% to 5%,Depot x PSKU,secondary
28,B2B,CNO,NHR-UTTAM,2025-11-30,M+1,2025-12-31,D231,718474,0.001261,0.001261,...,0.000032,0.000000,0.000032,0.000179,0.000675,0.000675,0.000000,0% to 5%,Depot x PSKU,secondary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320014,Ecom,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810805,0.000000,NaN,...,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%,Depot x PSKU,offtakes_to_primary
320015,Ecom,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810807,0.000000,NaN,...,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%,Depot x PSKU,offtakes_to_primary
320016,Ecom,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810919,0.000000,NaN,...,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%,Depot x PSKU,offtakes_to_primary
320017,Ecom,Male Grooming,SW_SGPRF,2026-03-31,M+1,2026-04-30,D677,811169,0.000000,NaN,...,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%,Depot x PSKU,offtakes_to_primary


In [681]:
# final_df[final_df['month'] == '2026-02-28'][['channel', 'asm', 'depot', 'psku', 'month', 'stat vol', 'consensus vol', 'actuals vol']].head(60)

In [123]:
final_df

,channel,portfolio,brand,brand class,run_month,m month,month,depot,psku,consensus vol,...,actuals val,stat error,consensus error,stat abs error,consensus abs error,forecast_granularity,forecast_type,index rate,stat_bias,stat_bias_bucket
0,B2B,CNO,NHR-UTTAM,B,2025-10-31,M+1,2025-11-30,D231,718474,0.0,...,0.000000,0.000243,0.000000,0.000243,0.000000,Depot x PSKU,secondary,NaN,NaN,NaN
1,B2B,CNO,NHR-UTTAM,B,2025-10-31,M+1,2025-11-30,D231,718475,0.0,...,0.000875,-0.000838,-0.000875,0.000838,0.000875,Depot x PSKU,secondary,NaN,NaN,NaN
2,B2B,CNO,NHR-UTTAM,B,2025-10-31,M+1,2025-11-30,D231,718476,0.0,...,0.000350,0.001727,-0.000350,0.001727,0.000350,Depot x PSKU,secondary,NaN,NaN,NaN
3,B2B,CNO,NHR-UTTAM,B,2025-10-31,M+1,2025-11-30,D232,718475,0.0,...,0.000000,0.001074,0.000000,0.001074,0.000000,Depot x PSKU,secondary,NaN,NaN,NaN
4,B2B,CNO,NHR-UTTAM,B,2025-11-30,M+1,2025-12-31,D231,718474,0.0,...,0.000000,0.000032,0.000000,0.000032,0.000000,Depot x PSKU,secondary,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320014,Ecom,Skin Care,PABABY_GM,NaN,2026-03-31,M+1,2026-04-30,D677,810805,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,Depot x PSKU,offtakes_to_primary,366.484998,0.0,0% to 5%
320015,Ecom,Skin Care,PABABY_GM,NaN,2026-03-31,M+1,2026-04-30,D677,810807,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,Depot x PSKU,offtakes_to_primary,366.484998,0.0,0% to 5%
320016,Ecom,Skin Care,PABABY_GM,NaN,2026-03-31,M+1,2026-04-30,D677,810919,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,Depot x PSKU,offtakes_to_primary,366.484998,0.0,0% to 5%
320017,Ecom,Male Grooming,SW_SGPRF,NaN,2026-03-31,M+1,2026-04-30,D677,811169,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,Depot x PSKU,offtakes_to_primary,1712.605337,0.0,0% to 5%


In [479]:
qcom_df['run_month'].unique()

<DatetimeArray>
['2025-11-30 00:00:00', '2025-12-31 00:00:00', '2026-01-31 00:00:00',
 '2026-02-28 00:00:00', '2026-03-31 00:00:00']
Length: 5, dtype: datetime64[ns]

In [481]:
final_df[final_df['run_month'].isin(['2026-03-31']) ].to_excel('/data/aman_singh/acuuracy_check/acc_framework_apr_final.xlsx')

In [686]:
final_df[(final_df['month'] >= '2026-01-31') & (final_df['month'] <= '2026-02-28')]['channel'].unique()

array(['B2B', 'ECOM', 'GT', 'MT', 'Qcom', 'Ecom'], dtype=object)

### offtakes

In [7]:
query = """select * from TRN_MIL_DF_OFFTAKES_OUTPUT"""
df_offtakes = pd.read_sql(query, dev_conn)
df_offtakes.columns = df_offtakes.columns.str.lower()
df_offtakes['run_month'] = pd.to_datetime(df_offtakes['run_month'])
df_offtakes['month_date'] = pd.to_datetime(df_offtakes['month_date'])
df_offtakes

,month_date,platform_name,parent_material_code,brand_code,portfolio,run_month,m month,pred_prophet,pred_rf,final_heuristic_prophet_value_2,channel
0,2026-01-31,Blinkit,718288,SAFF GOLD,Saffola Oils,2026-01-31,M,60.461823,58.398089,0.830145,QCOM
1,2026-01-31,Blinkit,718310,PCNO(R),CNO,2026-01-31,M,0.000000,0.000000,0.000790,QCOM
2,2026-01-31,Blinkit,718312,PCNO(R),CNO,2026-01-31,M,8.842138,6.024962,0.281155,QCOM
3,2026-01-31,Blinkit,718315,PCNO(R),CNO,2026-01-31,M,0.000000,0.000300,0.000000,QCOM
4,2026-01-31,Blinkit,718317,H&C,Hair Oils,2026-01-31,M,0.000000,0.049697,0.000000,QCOM
...,...,...,...,...,...,...,...,...,...,...,...
95506,2026-11-30,Zepto,810522,SAF_CDPRS,Saffola Oils,2026-03-31,M+8,0.000000,0.000000,0.003014,QCOM
95507,2026-11-30,Zepto,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+8,0.000000,0.000000,0.010053,QCOM
95508,2026-11-30,Zepto,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+8,0.000000,0.000000,0.007922,QCOM
95509,2026-11-30,Zepto,810685,SAF-MUSLI,Foods,2026-03-31,M+8,0.000000,0.000000,0.000580,QCOM


In [9]:
query_qcom = """select * from TRN_DF_QCOM_OFFTAKE_CHAIN_PSKU
where run_month = '2026-04-30'
and month_date between '2026-02-28' and '2026-03-31'
"""
qcom_df = pd.read_sql(query_qcom, dev_conn)
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month_date'] = pd.to_datetime(qcom_df['month_date'])
qcom_df['channel'] = 'QCOM'
qcom_df

,month_date,key,platform_name,parent_material_code,brand_code,vol_in_rum,run_month,imputed,channel
0,2026-02-28,Blinkit_718288,Blinkit,718288.0,SAFF GOLD,60.5760,2026-04-30,0,QCOM
1,2026-03-31,Blinkit_718288,Blinkit,718288.0,SAFF GOLD,77.6460,2026-04-30,0,QCOM
2,2026-02-28,Blinkit_718310,Blinkit,718310.0,PCNO(R),0.0005,2026-04-30,0,QCOM
3,2026-03-31,Blinkit_718310,Blinkit,718310.0,PCNO(R),0.0000,2026-04-30,0,QCOM
4,2026-02-28,Blinkit_718312,Blinkit,718312.0,PCNO(R),5.5660,2026-04-30,0,QCOM
...,...,...,...,...,...,...,...,...,...
1738,2026-02-28,Zepto_810738,Zepto,810738.0,PABABY_GM,120.1840,2026-04-30,0,QCOM
1739,2026-03-31,Zepto_810738,Zepto,810738.0,PABABY_GM,176.6560,2026-04-30,0,QCOM
1740,2026-03-31,Zepto_810971,Zepto,810971.0,PA_ESS_HO,0.3600,2026-04-30,0,QCOM
1741,2026-03-31,Zepto_811005,Zepto,811005.0,PA_ESS_HO,0.3080,2026-04-30,0,QCOM


In [11]:
query_ecom = """select * from TRN_DF_ECOM_OFFTAKE_CHAIN_PSKU
where run_month = '2026-04-30'
and month_date between '2026-02-28' and '2026-03-31'
"""
ecom_df = pd.read_sql(query_ecom, dev_conn)
ecom_df.columns = ecom_df.columns.str.lower()
ecom_df['run_month'] = pd.to_datetime(ecom_df['run_month'])
ecom_df['month_date'] = pd.to_datetime(ecom_df['month_date'])
ecom_df['channel'] = 'ECOM'
ecom_df

,month_date,platform_name,parent_material_code,brand_code,vol_in_rum,indexbpm,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,channel
0,2026-02-28,Amazon ARIPL,718288,SAFF GOLD,20.1780,27.777630,0,0,0,0,0,0,0,0,0,0,0,2026-04-30,ECOM
1,2026-03-31,Amazon ARIPL,718288,SAFF GOLD,39.3720,54.200652,0,0,0,0,0,0,0,0,0,0,0,2026-04-30,ECOM
2,2026-02-28,Amazon ARIPL,718321,SAFF KO,0.0000,0.000000,1,0,0,0,0,0,0,0,0,0,0,2026-04-30,ECOM
3,2026-03-31,Amazon ARIPL,718321,SAFF KO,0.0000,0.000000,1,0,0,0,0,0,0,0,0,0,0,2026-04-30,ECOM
4,2026-02-28,Amazon ARIPL,718322,SAFF KO,7.8050,13.256560,0,0,0,0,0,0,0,0,0,0,0,2026-04-30,ECOM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6470,2026-03-31,Purplle,809250,LVNPST_ML,0.0011,0.040458,0,0,0,0,0,0,0,0,0,0,0,2026-04-30,ECOM
6471,2026-02-28,Purplle,810673,PA_ESS_HO,0.1260,0.016250,0,0,0,0,0,0,0,0,0,0,0,2026-04-30,ECOM
6472,2026-03-31,Purplle,810673,PA_ESS_HO,0.0560,0.007224,0,0,0,0,0,0,0,0,0,0,0,2026-04-30,ECOM
6473,2026-02-28,Purplle,810674,PA_ESS_HO,0.2240,0.028900,0,0,0,0,0,0,0,0,0,0,0,2026-04-30,ECOM


In [12]:
offtake_actuals = pd.concat([qcom_df, ecom_df], ignore_index=True)
offtake_actuals

,month_date,key,platform_name,parent_material_code,brand_code,vol_in_rum,run_month,imputed,channel,indexbpm,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2
0,2026-02-28,Blinkit_718288,Blinkit,718288.0,SAFF GOLD,60.5760,2026-04-30,0,QCOM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-31,Blinkit_718288,Blinkit,718288.0,SAFF GOLD,77.6460,2026-04-30,0,QCOM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-02-28,Blinkit_718310,Blinkit,718310.0,PCNO(R),0.0005,2026-04-30,0,QCOM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-31,Blinkit_718310,Blinkit,718310.0,PCNO(R),0.0000,2026-04-30,0,QCOM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-02-28,Blinkit_718312,Blinkit,718312.0,PCNO(R),5.5660,2026-04-30,0,QCOM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8213,2026-03-31,NaN,Purplle,809250.0,LVNPST_ML,0.0011,2026-04-30,0,ECOM,0.040458,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8214,2026-02-28,NaN,Purplle,810673.0,PA_ESS_HO,0.1260,2026-04-30,0,ECOM,0.016250,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8215,2026-03-31,NaN,Purplle,810673.0,PA_ESS_HO,0.0560,2026-04-30,0,ECOM,0.007224,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8216,2026-02-28,NaN,Purplle,810674.0,PA_ESS_HO,0.2240,2026-04-30,0,ECOM,0.028900,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
df_offtakes = df_offtakes[df_offtakes['run_month'].isin(['2026-02-28', '2026-01-31'])]
df_offtakes = df_offtakes[df_offtakes['m month'] == 'M+1']
df_offtakes

,month_date,platform_name,parent_material_code,brand_code,portfolio,run_month,m month,pred_prophet,pred_rf,final_heuristic_prophet_value_2,channel
837,2026-02-28,Blinkit,718288,SAFF GOLD,Saffola Oils,2026-01-31,M+1,60.295965,57.833437,0.829005,QCOM
838,2026-02-28,Blinkit,718310,PCNO(R),CNO,2026-01-31,M+1,0.000000,0.000000,0.000790,QCOM
839,2026-02-28,Blinkit,718312,PCNO(R),CNO,2026-01-31,M+1,8.137921,5.823517,0.281155,QCOM
840,2026-02-28,Blinkit,718315,PCNO(R),CNO,2026-01-31,M+1,0.000000,0.000000,0.000000,QCOM
841,2026-02-28,Blinkit,718317,H&C,Hair Oils,2026-01-31,M+1,0.000000,0.049697,0.000000,QCOM
...,...,...,...,...,...,...,...,...,...,...,...
41544,2026-03-31,Nykaa,810605,KAYA_ML,Skin Care,2026-02-28,M+1,0.000000,0.135000,0.000000,ECOM
41545,2026-03-31,Nykaa,810673,PA_ESS_HO,Hair Oils,2026-02-28,M+1,0.000000,0.000000,0.002456,ECOM
41546,2026-03-31,Nykaa,810674,PA_ESS_HO,Hair Oils,2026-02-28,M+1,0.000000,0.000000,0.000933,ECOM
41547,2026-03-31,Nykaa,810738,PABABY_GM,Skin Care,2026-02-28,M+1,0.000000,0.000000,0.000000,ECOM


In [15]:
df_offtakes = df_offtakes.merge(
    offtake_actuals[['channel', 'platform_name', 'parent_material_code','month_date', 'vol_in_rum']]
    ,
    on=['channel', 'platform_name', 'parent_material_code','month_date'],
    how='left'
)

In [16]:
df_offtakes.rename(columns={'brand_code':'brand'}, inplace=True)

In [ ]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()
len_before_merge = len(df_offtakes)
df_offtakes = df_offtakes.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(df_offtakes)
del len_before_merge

NameError: name 'df' is not defined

In [20]:
df_offtakes.rename(columns={'month_date':'month', 'final_heuristic_prophet_value_2':'offtakes_forecasted_value'}, inplace=True)
df_offtakes['offtakes_forecasted_vol'] = df_offtakes['offtakes_forecasted_value'] * (10 ** 7) / df_offtakes['Index Rate']
df_offtakes['offtakes actuals val'] = df_offtakes['vol_in_rum'] * df_offtakes['Index Rate'] / (10 ** 7)
df_offtakes['offtakes error'] = df_offtakes['offtakes actuals val'] - df_offtakes['offtakes_forecasted_value'] 
df_offtakes['offtakes abs error'] = np.abs(df_offtakes['offtakes error'])
df_offtakes


,month,platform_name,parent_material_code,brand,portfolio,run_month,m month,pred_prophet,pred_rf,offtakes_forecasted_value,channel,vol_in_rum,Index Rate,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error
0,2026-02-28,Blinkit,718288,SAFF GOLD,Saffola Oils,2026-01-31,M+1,60.295965,57.833437,0.829005,QCOM,60.5760,138865.260689,59.698539,0.841190,0.012185,0.012185
1,2026-02-28,Blinkit,718310,PCNO(R),CNO,2026-01-31,M+1,0.000000,0.000000,0.000790,QCOM,0.0005,349274.001420,0.022616,0.000017,-0.000772,0.000772
2,2026-02-28,Blinkit,718312,PCNO(R),CNO,2026-01-31,M+1,8.137921,5.823517,0.281155,QCOM,5.5660,349274.001420,8.049687,0.194406,-0.086749,0.086749
3,2026-02-28,Blinkit,718315,PCNO(R),CNO,2026-01-31,M+1,0.000000,0.000000,0.000000,QCOM,0.0000,349274.001420,0.000000,0.000000,0.000000,0.000000
4,2026-02-28,Blinkit,718317,H&C,Hair Oils,2026-01-31,M+1,0.000000,0.049697,0.000000,QCOM,0.0000,388.076436,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8297,2026-03-31,Nykaa,810605,KAYA_ML,Skin Care,2026-02-28,M+1,0.000000,0.135000,0.000000,ECOM,0.0000,1226.374229,0.000000,0.000000,0.000000,0.000000
8298,2026-03-31,Nykaa,810673,PA_ESS_HO,Hair Oils,2026-02-28,M+1,0.000000,0.000000,0.002456,ECOM,1.7640,12860.631072,1.909829,0.002269,-0.000188,0.000188
8299,2026-03-31,Nykaa,810674,PA_ESS_HO,Hair Oils,2026-02-28,M+1,0.000000,0.000000,0.000933,ECOM,0.4060,12860.631072,0.725548,0.000522,-0.000411,0.000411
8300,2026-03-31,Nykaa,810738,PABABY_GM,Skin Care,2026-02-28,M+1,0.000000,0.000000,0.000000,ECOM,0.0000,366.484998,0.000000,0.000000,0.000000,0.000000


In [21]:
df_offtakes['offtakes_bias'] = df_offtakes['offtakes error']/df_offtakes['offtakes actuals val']
df_offtakes = df_offtakes.fillna(0)

df_offtakes['offtakes_bias'] = (
    df_offtakes['offtakes_bias']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
labels = [
    '< -15%',
    '-15% to -10%',
    '-10% to -5%',
    '-5% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 15%',
    '> 15%'
]

df_offtakes['offtakes_bias_bucket'] = pd.cut(
    df_offtakes['offtakes_bias'],
    bins=bins,
    labels=labels,
    right=False  
)


In [25]:
df_offtakes['forecast_granularity'] = 'Chain x PSKU'
df_offtakes['forecast_type'] = 'offtakes'

In [26]:
df_stat_fva = pd.read_excel('/data/aman_singh/acuuracy_check/fva_and_stat_accuracy.xlsx', sheet_name = 'fva_depot_psku_final')
df_stat_fva

,forecast_granularity,forecast_type,channel,depot,psku,brand,portfolio,month,m month,stat vol,...,dp error drm,dp abs error drm,dp error cam,dp abs error cam,dp error bam,dp abs error bam,Bias bucket drm,bias bucket cam,bias_bucket_bam,bias_bucket_plan
0,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-11-30,M+1,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
1,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-12-31,M+1,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
2,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-11-30,M+1,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
3,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-12-31,M+1,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
4,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2026-01-31,M+1,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142881,Depot x PSKU,offtakes_to_primary,QCOM,D677,810521,SAF_CDPRS,Saffola Oils,2026-03-31,M+1,0.012889,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
142882,Depot x PSKU,offtakes_to_primary,QCOM,D677,810522,SAF_CDPRS,Saffola Oils,2026-03-31,M+1,0.012509,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
142883,Depot x PSKU,offtakes_to_primary,QCOM,D677,810738,PABABY_GM,Skin Care,2026-03-31,M+1,11.106341,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
142884,Depot x PSKU,offtakes_to_primary,QCOM,D677,810805,PABABY_GM,Skin Care,2026-03-31,M+1,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%


In [28]:
df_offtakes.rename(columns={'platform_name':'chain', 'parent_material_code':'psku'
                            ,'Index Rate':'index rate'}, inplace=True)
df_offtakes.drop(['vol_in_rum','pred_prophet', 'pred_rf','run_month'], axis=1, inplace=True)
df_offtakes

,month,chain,psku,brand,portfolio,m month,offtakes_forecasted_value,channel,index rate,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket,forecast_granularity,forecast_type
0,2026-02-28,Blinkit,718288,SAFF GOLD,Saffola Oils,M+1,0.829005,QCOM,138865.260689,59.698539,0.841190,0.012185,0.012185,0.014485,0% to 5%,Chain x PSKU,offtakes
1,2026-02-28,Blinkit,718310,PCNO(R),CNO,M+1,0.000790,QCOM,349274.001420,0.022616,0.000017,-0.000772,0.000772,-44.231134,< -15%,Chain x PSKU,offtakes
2,2026-02-28,Blinkit,718312,PCNO(R),CNO,M+1,0.281155,QCOM,349274.001420,8.049687,0.194406,-0.086749,0.086749,-0.446225,< -15%,Chain x PSKU,offtakes
3,2026-02-28,Blinkit,718315,PCNO(R),CNO,M+1,0.000000,QCOM,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
4,2026-02-28,Blinkit,718317,H&C,Hair Oils,M+1,0.000000,QCOM,388.076436,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8297,2026-03-31,Nykaa,810605,KAYA_ML,Skin Care,M+1,0.000000,ECOM,1226.374229,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
8298,2026-03-31,Nykaa,810673,PA_ESS_HO,Hair Oils,M+1,0.002456,ECOM,12860.631072,1.909829,0.002269,-0.000188,0.000188,-0.082669,-10% to -5%,Chain x PSKU,offtakes
8299,2026-03-31,Nykaa,810674,PA_ESS_HO,Hair Oils,M+1,0.000933,ECOM,12860.631072,0.725548,0.000522,-0.000411,0.000411,-0.787063,< -15%,Chain x PSKU,offtakes
8300,2026-03-31,Nykaa,810738,PABABY_GM,Skin Care,M+1,0.000000,ECOM,366.484998,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes


In [29]:
final_df = pd.concat([df_stat_fva, df_offtakes], ignore_index=True)
final_df

,forecast_granularity,forecast_type,channel,depot,psku,brand,portfolio,month,m month,stat vol,...,bias_bucket_bam,bias_bucket_plan,chain,offtakes_forecasted_value,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket
0,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-11-30,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-12-31,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-11-30,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-12-31,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2026-01-31,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151183,Chain x PSKU,offtakes,ECOM,NaN,810605,KAYA_ML,Skin Care,2026-03-31,M+1,NaN,...,NaN,NaN,Nykaa,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
151184,Chain x PSKU,offtakes,ECOM,NaN,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+1,NaN,...,NaN,NaN,Nykaa,0.002456,1.909829,0.002269,-0.000188,0.000188,-0.082669,-10% to -5%
151185,Chain x PSKU,offtakes,ECOM,NaN,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+1,NaN,...,NaN,NaN,Nykaa,0.000933,0.725548,0.000522,-0.000411,0.000411,-0.787063,< -15%
151186,Chain x PSKU,offtakes,ECOM,NaN,810738,PABABY_GM,Skin Care,2026-03-31,M+1,NaN,...,NaN,NaN,Nykaa,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%


In [30]:
final_df.columns

Index(['forecast_granularity', 'forecast_type', 'channel', 'depot', 'psku',
       'brand', 'portfolio', 'month', 'm month', 'stat vol', 'actuals vol',
       'consensus vol', 'index rate', 'stat val', 'consensus val',
       'actuals val', 'stat error', 'consensus error', 'stat abs error',
       'consensus abs error', 'stat_bias', 'stat_bias_bucket',
       'delivery_vol_drm', 'delivery val_drm', 'delivery_vol_cam',
       'delivery val_cam', 'delivery_vol_bam', 'delivery val_bam',
       'brand class', 'dp error drm', 'dp abs error drm', 'dp error cam',
       'dp abs error cam', 'dp error bam', 'dp abs error bam',
       'Bias bucket drm', 'bias bucket cam', 'bias_bucket_bam',
       'bias_bucket_plan', 'chain', 'offtakes_forecasted_value',
       'offtakes_forecasted_vol', 'offtakes actuals val', 'offtakes error',
       'offtakes abs error', 'offtakes_bias', 'offtakes_bias_bucket'],
      dtype='object')

In [32]:
final_df.to_csv('/data/aman_singh/acuuracy_check/fva_stat_offtake_accuracy.csv', index=False)

### Chain psku primary accuracy

In [7]:
qcom_df = pd.read_sql(
    """select * from TRN_MIL_DF_OFT2PRIM_CPSKU""",
    dev_conn
)
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month'] = pd.to_datetime(qcom_df['month'])

In [8]:
qcom_df#[qcom_df['Channel']=='Ecom']

,month,chain,psku,brand,portfolio,m month,run_month,calculated primary vol,channel
0,2025-12-31,Blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.0000,Qcom
1,2025-12-31,Blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.6195,Qcom
2,2025-12-31,Blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.0000,Qcom
3,2025-12-31,Blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.0000,Qcom
4,2025-12-31,Blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.0000,Qcom
...,...,...,...,...,...,...,...,...,...
42476,2026-06-30,Nykaa,810805,PABABY_GM,Skin Care,M+4,2026-02-28,0.0000,Ecom
42477,2026-06-30,Nykaa,810807,PABABY_GM,Skin Care,M+4,2026-02-28,0.0000,Ecom
42478,2026-06-30,Nykaa,810919,PABABY_GM,Skin Care,M+4,2026-02-28,0.0000,Ecom
42479,2026-06-30,Nykaa,811169,SW_SGPRF,Male Grooming,M+4,2026-02-28,0.0000,Ecom


In [9]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

# realignment_df = realignment_df[
#     realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column, channel='QCOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        realignment_data['channel'].isin([channel, channel + ' B2C', 'ALL'])
    ]

    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [17]:
chain_psku_primary_query = """
SELECT
    CASE
        WHEN MCM.chain = 'Kiranakart Technologies' THEN 'Zepto'
        WHEN MCM.chain = 'Grofers' THEN 'Blinkit'
        ELSE MCM.chain
    END AS chain,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Grofers', 'Zepto', 'Kiranakart Technologies', 'Swiggy', 'ZEPTO')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
where month_date BETWEEN '2025-12-01' AND '2026-03-31' 
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    chain_psku_primary_query,
    prod_conn
)
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)
depot_psku_primary_df['month_date'] = pd.to_datetime(depot_psku_primary_df['month_date'])
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code')
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['chain', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]
depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [19]:
depot_psku_primary_df[depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date'], keep=False)]

,chain,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
651,Blinkit,732296,PABABY_ML,2025-12-31,0.00,129.142776,55.5023,0.00
652,Blinkit,732296,PABABY_ML,2026-01-31,0.00,161.650358,116.1056,0.00
653,Blinkit,732296,PABABY_SP,2025-12-31,33.60,0.000000,0.0000,33.60
654,Blinkit,732296,PABABY_SP,2026-01-31,0.00,0.000000,0.0000,0.00
657,Blinkit,732297,PABABY_ML,2025-12-31,0.00,155.915998,100.1441,0.00
658,Blinkit,732297,PABABY_ML,2026-01-31,0.00,397.753905,283.5170,0.00
659,Blinkit,732297,PABABY_ML,2026-02-28,0.00,0.000000,0.0000,0.00
660,Blinkit,732297,PABABY_SP,2025-12-31,137.76,0.000000,0.0000,137.76
661,Blinkit,732297,PABABY_SP,2026-01-31,58.22,0.000000,0.0000,58.22
662,Blinkit,732297,PABABY_SP,2026-02-28,39.36,230.724898,209.3269,39.36


In [20]:
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['material_group_code']!='PABABY_SP']
depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()


0

In [21]:
depot_psku_primary_df

,chain,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,Blinkit,718288,SAFF GOLD,2025-12-31,66.186,77.024780,76.6195,66.186
1,Blinkit,718288,SAFF GOLD,2026-01-31,51.666,49.549496,52.9887,51.666
2,Blinkit,718288,SAFF GOLD,2026-02-28,61.632,55.342992,60.4012,61.632
3,Blinkit,718288,SAFF GOLD,2026-03-31,61.434,34.113398,67.0527,61.434
4,Blinkit,718299,PCNO(R),2026-01-31,0.000,0.742105,0.0000,0.000
...,...,...,...,...,...,...,...,...
2672,Zepto,811005,PA_ESS_HO,2026-02-28,6.048,0.000000,0.0000,6.048
2673,Zepto,811169,SW_SGPRF,2026-02-28,0.000,0.000000,120.6350,0.000
2674,Zepto,811169,SW_SGPRF,2026-03-31,0.000,96.500506,148.9285,0.000
2675,Zepto,811181,SAF_CDPRS,2026-02-28,0.408,1.199970,1.3334,0.408


In [22]:
depot_psku_primary_df['channel'] = 'Qcom'
depot_psku_primary_df_qcom = depot_psku_primary_df.copy()

In [24]:
depot_psku_primary_query = """
SELECT
    CASE 
        WHEN MCM.chain = 'Big basket B2C' THEN 'Big Basket'
        WHEN MCM.chain = 'Flipkart-National' THEN 'Flipkart National'
        WHEN MCM.chain = 'Flipkart-Minutes' THEN 'Flipkart National'
        WHEN MCM.chain = 'Nykaa' THEN 'Nykaa'
        WHEN MCM.chain = 'Purplle' THEN 'Purplle'
        WHEN MCM.chain = 'Myntra' THEN 'Myntra'
        WHEN MCM.chain = 'MYNTRA' THEN 'Myntra'
        WHEN MCM.chain = 'Amazon B2C' THEN 'Amazon ARIPL'
        WHEN MCM.chain = 'ARIPL' THEN 'Amazon ARIPL'
        WHEN MCM.chain = 'RK WORLDINFOCOM' THEN 'Amazon RK'
        WHEN MCM.chain = 'RKWorld' THEN 'Amazon RK'
        WHEN MCM.chain = 'Flipkart-Grocery' THEN 'Flipkart Grocery'
        WHEN MCM.chain = 'FlipkartGrocery' THEN 'Flipkart Grocery'
        WHEN MCM.chain = 'Firstcry' THEN 'First Cry'
        WHEN MCM.chain = 'CITIMALL' THEN 'City Mall'
        ELSE MCM.chain
    END AS chain,
    MM.parent_material_code,
    MM.material_group_code,
    MESR.month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Flipkart-National', 'Flipkart-Grocery', 'Big basket B2C', 'RK WORLDINFOCOM', 'Amazon B2C', 
        'FATEHPURIA HYGIENE', 'Nykaa', 'Flipkart-Minutes', 'Purplle', 'Myntra', 'Dealshare', 'FlipkartGrocery', 'City Mall', '1MG', 
        'ARIPL', 'First Cry', 'Meesho', 'RKWorld', 'CITIMALL', 'Firstcry', 'EMAZING DEALS', 'MYNTRA')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
WHERE
    MESR.month_date between '2025-12-31' and '2026-03-31'
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""
depot_psku_primary_df = pd.read_sql(
    depot_psku_primary_query,
    prod_conn
)
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)
depot_psku_primary_df['month_date'] = pd.to_datetime(depot_psku_primary_df['month_date'])
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code', channel='ECOM')
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['chain', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]
depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [ ]:
depot_psku_primary_df['channel'] = 'Ecom'
depot_psku_primary_df_ecom = depot_psku_primary_df.copy()

In [ ]:
depot_psku_primary_df = pd.concat([depot_psku_primary_df_qcom, depot_psku_primary_df_ecom], ignore_index=True)
depot_psku_primary_df

,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,channel
0,D112,718310,PCNO(R),2026-01-31,0.00,0.012857,0.0000,0.00,Qcom
1,D112,718312,PCNO(R),2025-12-31,0.46,0.958689,1.1501,0.46,Qcom
2,D112,718312,PCNO(R),2026-01-31,0.24,1.079865,1.4001,0.24,Qcom
3,D112,718312,PCNO(R),2026-02-28,0.56,0.827936,0.9996,0.56,Qcom
4,D112,718318,H&C,2026-01-31,14.40,0.000000,0.0000,14.40,Qcom
...,...,...,...,...,...,...,...,...,...
27532,D677,810971,PA_ESS_HO,2026-02-28,0.00,0.370378,0.0000,0.00,Ecom
27533,D677,811005,PA_ESS_HO,2025-12-31,0.00,0.266666,0.0000,0.00,Ecom
27534,D677,811005,PA_ESS_HO,2026-01-31,0.00,0.240200,0.0000,0.00,Ecom
27535,D677,811005,PA_ESS_HO,2026-02-28,0.00,0.268800,0.0000,0.00,Ecom


In [ ]:
actuals_df = depot_psku_primary_df.copy()

In [ ]:
actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date']).sum()

0

In [ ]:
actuals_df = actuals_df.groupby(['channel','depot_code', 'parent_material_code',
       'material_group_code', 'month_date'])[['pri_actuals_vol_rum',
       'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum().reset_index()
actuals_df

,channel,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,Ecom,D112,718287,PCNO(R),2025-12-31,0.000,0.023463,0.0000,0.000
1,Ecom,D112,718308,PCNO(R),2025-12-31,0.126,0.000000,0.0000,0.126
2,Ecom,D112,718308,PCNO(R),2026-01-31,0.036,0.000000,0.0000,0.036
3,Ecom,D112,718312,PCNO(R),2025-12-31,0.720,0.182709,0.1849,0.720
4,Ecom,D112,718312,PCNO(R),2026-01-31,0.340,0.696102,0.7101,0.340
...,...,...,...,...,...,...,...,...,...
27532,Qcom,D677,810738,PABABY_GM,2026-01-31,17.376,0.000000,0.0000,17.376
27533,Qcom,D677,810738,PABABY_GM,2026-02-28,8.688,0.000000,0.0000,8.688
27534,Qcom,D677,810971,PA_ESS_HO,2026-01-31,0.000,0.466326,0.0000,0.000
27535,Qcom,D677,810971,PA_ESS_HO,2026-02-28,0.000,0.000150,0.0000,0.000


In [ ]:
actuals_df.columns

Index(['channel', 'depot_code', 'parent_material_code', 'material_group_code',
       'month_date', 'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum',
       'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum'],
      dtype='object')

In [ ]:
actuals_df = actuals_df.rename(columns={
    'depot_code': 'depot',
    'parent_material_code': 'psku',
    'sec_apo_plan_vol_rum': 'Consensus Vol',
    'sec_actuals_vol_rum': 'Actuals Vol',
    'month_date': 'month'
})

In [ ]:
actuals_df.head()

,channel,depot,psku,material_group_code,month,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol
0,Ecom,D112,718287,PCNO(R),2025-12-31,0.000,0.023463,0.0000,0.000
1,Ecom,D112,718308,PCNO(R),2025-12-31,0.126,0.000000,0.0000,0.126
2,Ecom,D112,718308,PCNO(R),2026-01-31,0.036,0.000000,0.0000,0.036
3,Ecom,D112,718312,PCNO(R),2025-12-31,0.720,0.182709,0.1849,0.720
4,Ecom,D112,718312,PCNO(R),2026-01-31,0.340,0.696102,0.7101,0.340


In [ ]:
#qcom_df.rename(columns = {'Channel':'channel'}, inplace = True)

In [ ]:
qcom_df['psku'] = qcom_df['psku'].astype(int)
actuals_df['psku'] = actuals_df['psku'].astype(int)

In [ ]:
# actuals_df = actuals_df[actuals_df['month_date'] == '2025-12-31']
# actuals_df

In [ ]:
len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    actuals_df.drop([ 'material_group_code', 'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum'], axis=1),
    on=['channel','depot', 'psku', 'month'],
    how='left'
)
assert len_before_merge == len(qcom_df)
del len_before_merge

In [ ]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2026-03-31,PADV_WIPS,168.752000
1,2026-03-31,PABABY_SP,415.245000
2,2026-03-31,P_GOHR_SR,7470.000000
3,2026-03-31,PA_JAS_GD,298.847906
4,2026-03-31,PA_NOR_HO,188992.000000


In [ ]:
len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(qcom_df)
del len_before_merge

In [ ]:
qcom_df.isna().sum()

month                          0
depot                          0
psku                           0
brand                          0
portfolio                      0
m month                        0
run_month                      0
calculated primary vol         0
channel                        0
Consensus Vol             161500
Actuals Vol               161500
Index Rate                     0
dtype: int64

In [ ]:
qcom_df['Consensus Vol'] = qcom_df['Consensus Vol'].fillna(0)
qcom_df['Actuals Vol'] = qcom_df['Actuals Vol'].fillna(0)

In [ ]:
qcom_df['Consensus Vol'].max()

16869.7071

In [ ]:
qcom_df.rename(columns={'calculated primary vol': 'Stat Vol'}, inplace=True)
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate
0,2025-11-30,0x2a,718287,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,309765.865129
1,2025-11-30,0x2a,718288,SAFF GOLD,Saffola Oils,M,2025-11-30,0.0,Qcom,0.0,0.0,137662.938527
2,2025-11-30,0x2a,718297,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,309765.865129
3,2025-11-30,0x2a,718299,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,309765.865129
4,2025-11-30,0x2a,718300,PCNO FLEX,CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,188992.272896
...,...,...,...,...,...,...,...,...,...,...,...,...
199425,2026-03-31,D677,810805,PABABY_GM,Skin Care,M+2,2026-01-31,0.0,Ecom,0.0,0.0,451.133000
199426,2026-03-31,D677,810919,PABABY_GM,Skin Care,M+2,2026-01-31,0.0,Ecom,0.0,0.0,451.133000
199427,2026-03-31,D677,810971,PA_ESS_HO,Hair Oils,M+2,2026-01-31,0.0,Ecom,0.0,0.0,12900.000000
199428,2026-03-31,D677,811005,PA_ESS_HO,Hair Oils,M+2,2026-01-31,0.0,Ecom,0.0,0.0,12900.000000


In [ ]:
qcom_df['Stat Val'] = qcom_df['Stat Vol'] * qcom_df['Index Rate'] / (10 ** 7)
qcom_df['Consensus Val'] = qcom_df['Consensus Vol'] * qcom_df['Index Rate'] / (10 ** 7)
qcom_df['Actuals Val'] = qcom_df['Actuals Vol'] * qcom_df['Index Rate'] / (10 ** 7)

In [ ]:
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate,Stat Val,Consensus Val,Actuals Val
0,2025-11-30,0x2a,718287,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,309765.865129,0.0,0.0,0.0
1,2025-11-30,0x2a,718288,SAFF GOLD,Saffola Oils,M,2025-11-30,0.0,Qcom,0.0,0.0,137662.938527,0.0,0.0,0.0
2,2025-11-30,0x2a,718297,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,309765.865129,0.0,0.0,0.0
3,2025-11-30,0x2a,718299,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,309765.865129,0.0,0.0,0.0
4,2025-11-30,0x2a,718300,PCNO FLEX,CNO,M,2025-11-30,0.0,Qcom,0.0,0.0,188992.272896,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199425,2026-03-31,D677,810805,PABABY_GM,Skin Care,M+2,2026-01-31,0.0,Ecom,0.0,0.0,451.133000,0.0,0.0,0.0
199426,2026-03-31,D677,810919,PABABY_GM,Skin Care,M+2,2026-01-31,0.0,Ecom,0.0,0.0,451.133000,0.0,0.0,0.0
199427,2026-03-31,D677,810971,PA_ESS_HO,Hair Oils,M+2,2026-01-31,0.0,Ecom,0.0,0.0,12900.000000,0.0,0.0,0.0
199428,2026-03-31,D677,811005,PA_ESS_HO,Hair Oils,M+2,2026-01-31,0.0,Ecom,0.0,0.0,12900.000000,0.0,0.0,0.0


In [ ]:
qcom_df[(qcom_df['month'] == '2026-02-28') & (qcom_df['channel']=='Ecom') & (qcom_df['run_month']=='2026-01-31')]['Stat Val'].sum()

34.48391685166551

In [ ]:
qcom_df['Stat Error'] = qcom_df['Stat Val'] - qcom_df['Actuals Val']
qcom_df['Consensus Error'] = qcom_df['Consensus Val'] - qcom_df['Actuals Val']

qcom_df['Stat Abs Error'] = np.abs(qcom_df['Stat Error'])
qcom_df['Consensus Abs Error'] = np.abs(qcom_df['Consensus Error'])

In [ ]:
qcom_df['stat_bias'] = qcom_df['Stat Error']/qcom_df['Actuals Val']
qcom_df = qcom_df.fillna(0)

import numpy as np
import pandas as pd

qcom_df['stat_bias'] = (
    qcom_df['stat_bias']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
labels = [
    '< -15%',
    '-15% to -10%',
    '-10% to -5%',
    '-5% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 15%',
    '> 15%'
]

qcom_df['stat_bias_bucket'] = pd.cut(
    qcom_df['stat_bias'],
    bins=bins,
    labels=labels,
    right=False  
)


In [ ]:
# delivery_df = pd.read_csv('/data/aman_singh/acuuracy_check/Export View of Month Vol & APO.csv')

# delivery_df['Channel'] = delivery_df['Channel'].replace({
#     'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
# delivery_df
# delivery_df['delivery_vol'] = delivery_df['01-12-2025 Del Vol'].map(lambda x:0 if x.strip() == '-' else float(x.strip().replace(',','')))
# delivery_df = delivery_df[delivery_df['Channel'] == 'QCOM']
# delivery_df = delivery_df.groupby(['Depot','PSKU'])['delivery_vol'].sum().reset_index(
# )#.rename(columns = {'01-12-2025 Del Vol':'delivery_vol'})

# len_before_merge = len(qcom_df)
# qcom_df = qcom_df.merge(
#     delivery_df,
#     on=['Depot', 'PSKU'],
#     how='left'
# )
# assert len_before_merge == len(qcom_df)
# del len_before_merge

# qcom_df['Delivery Val'] = qcom_df['delivery_vol'] * qcom_df['Index Rate'] / (10 ** 7)

# qcom_df['Dp Error'] = qcom_df['Delivery Val'] - qcom_df['Actuals Val']

# qcom_df['Dp Abs Error'] = np.abs(qcom_df['Dp Error'])



In [ ]:
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,stat vol,channel,consensus vol,...,index rate,stat val,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket
0,2025-11-30,0x2a,718287,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,...,309765.865129,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
1,2025-11-30,0x2a,718288,SAFF GOLD,Saffola Oils,M,2025-11-30,0.0,Qcom,0.0,...,137662.938527,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
2,2025-11-30,0x2a,718297,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,...,309765.865129,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
3,2025-11-30,0x2a,718299,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,...,309765.865129,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
4,2025-11-30,0x2a,718300,PCNO FLEX,CNO,M,2025-11-30,0.0,Qcom,0.0,...,188992.272896,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199425,2026-03-31,D677,810805,PABABY_GM,Skin Care,M+2,2026-01-31,0.0,Ecom,0.0,...,451.133000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
199426,2026-03-31,D677,810919,PABABY_GM,Skin Care,M+2,2026-01-31,0.0,Ecom,0.0,...,451.133000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
199427,2026-03-31,D677,810971,PA_ESS_HO,Hair Oils,M+2,2026-01-31,0.0,Ecom,0.0,...,12900.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
199428,2026-03-31,D677,811005,PA_ESS_HO,Hair Oils,M+2,2026-01-31,0.0,Ecom,0.0,...,12900.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%


In [ ]:
# df = pd.read_excel("/data/aman_singh/acuuracy_check/acc_framework_feb.xlsx")
# df.columns = df.columns.str.lower()
# df = df[df['forecast_granularity']!='Depot x PSKU']

In [ ]:
df.columns = df.columns.str.lower()

In [ ]:
qcom_df.columns

Index(['month', 'depot', 'psku', 'brand', 'portfolio', 'm month', 'run_month',
       'stat vol', 'channel', 'consensus vol', 'actuals vol', 'index rate',
       'stat val', 'consensus val', 'actuals val', 'stat error',
       'consensus error', 'stat abs error', 'consensus abs error', 'stat_bias',
       'stat_bias_bucket'],
      dtype='object')

In [ ]:

qcom_df['forecast_granularity'] = 'Depot x PSKU'
qcom_df['forecast_type'] = 'offtakes_to_primary'
#qcom_df['channel'] = 'QCOM'
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,stat vol,channel,consensus vol,...,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket,forecast_granularity,forecast_type
0,2025-11-30,0x2a,718287,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
1,2025-11-30,0x2a,718288,SAFF GOLD,Saffola Oils,M,2025-11-30,0.0,Qcom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
2,2025-11-30,0x2a,718297,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
3,2025-11-30,0x2a,718299,PCNO(R),CNO,M,2025-11-30,0.0,Qcom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
4,2025-11-30,0x2a,718300,PCNO FLEX,CNO,M,2025-11-30,0.0,Qcom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199425,2026-03-31,D677,810805,PABABY_GM,Skin Care,M+2,2026-01-31,0.0,Ecom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
199426,2026-03-31,D677,810919,PABABY_GM,Skin Care,M+2,2026-01-31,0.0,Ecom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
199427,2026-03-31,D677,810971,PA_ESS_HO,Hair Oils,M+2,2026-01-31,0.0,Ecom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
199428,2026-03-31,D677,811005,PA_ESS_HO,Hair Oils,M+2,2026-01-31,0.0,Ecom,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary


In [ ]:
qcom_df = qcom_df[qcom_df['m month'] == 'M+1']

In [ ]:
final_df = pd.concat([df,qcom_df])
final_df

,forecast_granularity,forecast_type,channel,portfolio,brand,brand class,run_month,m month,month,asm,...,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value,stat_bias,stat_bias_bucket,delivery_vol,delivery val,dp error,dp abs error
0,ASM x Depot x PSKU,secondary,B2B,CNO,NHR-UTTAM,B,2025-10-31,M+1,2025-11-30,BCE1,...,0.000188,0.000188,0.000207,0.000575,0.000000,0% to 5%,NaN,NaN,NaN,NaN
1,ASM x Depot x PSKU,secondary,B2B,CNO,NHR-UTTAM,B,2025-10-31,M+1,2025-11-30,BCE1,...,0.000028,0.000616,0.000031,0.001448,-0.958118,< -15%,NaN,NaN,NaN,NaN
2,ASM x Depot x PSKU,secondary,B2B,CNO,NHR-UTTAM,B,2025-10-31,M+1,2025-11-30,BCE1,...,0.002436,0.001722,0.000526,0.000809,4.933610,> 15%,NaN,NaN,NaN,NaN
3,ASM x Depot x PSKU,secondary,B2B,CNO,NHR-UTTAM,B,2025-10-31,M+1,2025-11-30,BCE1,...,0.000915,0.000370,0.000000,0.000000,0.000000,0% to 5%,NaN,NaN,NaN,NaN
4,ASM x Depot x PSKU,secondary,B2B,CNO,NHR-UTTAM,B,2025-11-30,M+1,2025-12-31,BCE1,...,0.000027,0.000152,0.000575,0.000575,0.000000,0% to 5%,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182265,Depot x PSKU,offtakes_to_primary,Ecom,Skin Care,PABABY_GM,NaN,2026-01-31,M+1,2026-02-28,NaN,...,NaN,NaN,NaN,NaN,0.000000,0% to 5%,NaN,NaN,NaN,NaN
182266,Depot x PSKU,offtakes_to_primary,Ecom,Skin Care,PABABY_GM,NaN,2026-01-31,M+1,2026-02-28,NaN,...,NaN,NaN,NaN,NaN,0.000000,0% to 5%,NaN,NaN,NaN,NaN
182267,Depot x PSKU,offtakes_to_primary,Ecom,Hair Oils,PA_ESS_HO,NaN,2026-01-31,M+1,2026-02-28,NaN,...,NaN,NaN,NaN,NaN,0.000000,0% to 5%,NaN,NaN,NaN,NaN
182268,Depot x PSKU,offtakes_to_primary,Ecom,Hair Oils,PA_ESS_HO,NaN,2026-01-31,M+1,2026-02-28,NaN,...,NaN,NaN,NaN,NaN,0.000000,0% to 5%,NaN,NaN,NaN,NaN


In [7]:
import pickle
with open('/data/aman_singh/acuuracy_check/prophet_models (2).pkl', 'rb') as f:
    model = pickle.load(f)

In [6]:
model['ORS_D535_731588']

{'changepoint_prior_scale': 0.01,
 'changepoint_range': 0.8,
 'seasonality_prior_scale': 2.0,
 'n_changepoints': 4,
 'yearly_seasonality': 5}

In [8]:
model['ORS_D535_731588']

{'changepoint_prior_scale': 0.01,
 'changepoint_range': 0.8,
 'seasonality_prior_scale': 0.1,
 'n_changepoints': 4,
 'yearly_seasonality': 4}

In [2]:
print(model.growth)

AttributeError: 'dict' object has no attribute 'growth'